# Recipe → Catalog matching: ghi chép pipeline (issue #8)

Notebook ghi lại **pipeline đã thống nhất**, phần **đã làm**, code giải thích, kết quả từng bước, **tự đánh giá** của AI và **hướng nâng cấp**.
Các cell code gọi lại đúng code trong `scripts/`, nên kết quả hiển thị là kết quả chạy thật trên dữ liệu hiện có.

> Mục tiêu cuối: người dùng gõ một đoạn text, hệ thống trả công thức giống nhất (top 1 nổi bật, 4 cái còn lại là lựa chọn khác),
> kèm một giỏ hàng nạp sẵn các nguyên liệu có trong catalog. Bấm một lần là bỏ cả giỏ vào.
> Giỏ phải **đủ nguyên liệu** và **đủ số lượng**.

| Bước | Nội dung | Trạng thái |
| :-- | :-- | :-: |
| 1 | Chuẩn hóa tên nguyên liệu trong công thức (`recipes_clean`) | ✅ làm xong |
| 2 | Chuẩn hóa catalog bằng cùng bộ luật (`catalog_clean`) | ✅ làm xong |
| 3 | bge-m3 gợi ý gộp tên đồng nghĩa và ghép nguyên liệu → catalog | ✅ đã chạy xong; ⏳ **chưa duyệt tay** |
| 4 | Chọn tập công thức mục tiêu (100 món), liệt kê hàng catalog còn thiếu | ✅ |
| 5 | Duyệt tay nguyên liệu → sản phẩm, bảng ghép chính thức, catalog v2 | ✅ (pantry mới là AI gợi ý, **chưa duyệt**) |
| 6 | Chốt catalog, **khóa catalog_id**, file bàn giao cho bên web | ✅ |
| 7 | Bảng quy đổi đơn vị | ❌ **bỏ** (quy tắc: mua nguyên gói theo đơn vị catalog) |
| 8 | Hạ tầng issue #8: `ai_db` (Docker + pgvector), torch GPU, `requirements` | ✅ |
| 9 | Nạp 2.527 công thức vào `ai_db`, embed 2 loại chunk, thử truy vấn (mục 11) | ✅ (tìm kiếm tốt trên toàn bộ công thức; **chưa đo Recall@5**) |
| 10 | Phần còn lại của issue #8: bộ truy vấn có nhãn + Recall@5, FastAPI `/health`, README | ⬜ **chưa làm** |
| 11 | Giỏ hàng cho món phổ biến (hiện chỉ 100 món có giỏ, **chưa có món canh chua nào**) | ⬜ phát hiện ở mục 11, cần quyết định |

## 0. Các quyết định đã thống nhất

*Cập nhật lần cuối 2026-10-24. Những mục đã thay đổi so với bản đầu được ghi rõ.*

- **Ghép offline.** Nguyên liệu → sản phẩm catalog được tính trước, kèm `catalog_id`. Lúc chạy chỉ embed câu hỏi và lấy top-5 công thức. *(Khác proposal: proposal ghép online bằng embedding và ngưỡng δ. Cần ghi vào `docs/06`.)*
- **Catalog là nguồn sự thật duy nhất**, không đổi đơn vị catalog. `catalog_id` (CAT-NNNNNN) **khóa cứng**, chỉ nối thêm.
- **Số lượng: mua nguyên gói.** *(Đã đổi so với bản đầu: không làm bảng quy đổi đơn vị.)* Mặc định 1 sản phẩm mỗi dòng; chỉ khi công thức ghi g/kg/ml/lít và vượt quy cách thì `packs` > 1. Khách tự tăng số lượng, AI chỉ gợi ý.
- **Danh sách nguyên liệu chia hai phần:** `buy` (nguyên liệu chính, mặc định 1) và `maybe_have` (đồ trong tủ bếp và đồ khách đã có, mặc định 0). Bấm "thêm vào giỏ" thì mọi dòng có số lượng > 0 vào giỏ.
- **Dạng nguyên liệu:** dạng *khó làm* (khô, đông lạnh, đóng hộp, ngâm chua) là mặt hàng riêng ("tôm khô", "cà chua hộp"). Dạng *dễ làm* (phi, xay, rang) gộp về tên gốc, ghi ở trường `form`.
- **Món nhỏ tự làm** (nước me, mỡ hành, vỏ chanh mài…) không thêm sản phẩm; trỏ về nguyên liệu gốc và đánh dấu `derived` để AI hướng dẫn làm thêm.
- **Màu ớt giữ riêng** (ớt đỏ ≠ ớt xanh). Bỏ các cụm sơ chế ("bóc vỏ", "pha loãng", "ngâm nở"...).
- **Phạm vi giỏ hàng:** *(Đã đổi: ban đầu chỉ 100 món.)* 1.061 món (100 món mục tiêu + 106 họ món phổ biến), đều đủ nguyên liệu chính.
- **Tìm món = hybrid:** từ khóa bỏ dấu + bge-m3; ngưỡng "không tìm thấy" khoảng 0.64 (mục 14).
- **ai-service lo trọn phần AI** (tìm món, lớp lời Gemini/Claude, kiểm tra, dự phòng); đường đi Frontend → Medusa backend → ai-service. Giỏ hàng luôn do code dựng. *(Đã đổi: bản đầu ghi "chưa dùng Gemini".)* Chi tiết ở mục 15 và `docs/recipe_rag/plan-ai-copilot.md`.
- **Hoãn:** dị ứng, ngân sách, nhóm sản phẩm thay thế tốt hơn, các bước cần web hoàn chỉnh (mục 16).

```
recipes.json ─► [1] chuẩn hóa tên ─► recipes_clean.json ──┐
catalog.json ─► [2] chuẩn hóa catalog ─► catalog_clean.json ┤
                       [3] bge-m3 gợi ý gộp/ghép ─► duyệt tay ─► bảng ghép + khóa catalog_id (mục 8, 9)
                       [4] mở rộng catalog cho món phổ biến, AI duyệt sơ bộ (mục 12) ─► catalog v2, giỏ 1.061 món
                       [5] ai_db: embed thẻ món + nạp giỏ/catalog (mục 10, 11, 15.1)
                       [6] ai-service: hybrid search ─► giỏ hai phần ─► lớp lời AI ─► (route HTTP, chờ web)  (mục 15)
```

In [1]:
import sys, json, re, collections
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))   # để import code của pipeline

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
REVIEW = ROOT / "data" / "review"

def table(rows, header):
    # In bảng markdown cho dễ đọc
    from IPython.display import Markdown, display
    lines = ["| " + " | ".join(header) + " |", "|" + " :--|" * len(header)]
    lines += ["| " + " | ".join(str(c).replace("|", "\\|") for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

## 1. Dữ liệu thô và vì sao phải chuẩn hóa

Hai nguồn: `recipes.json` (~2.500 món crawl từ monngonmoingay.com) và `products_clean_recipes.json` (catalog đã lọc ra hàng nấu ăn được).
Hai bên **không nói cùng một ngôn ngữ**: công thức ghi tên chung ("cà rốt", "thịt bò"), catalog ghi mặt hàng cụ thể ("Cà rốt 300g", "Thịt bò hai lát Vissan hộp 170g").

In [2]:
recipes_raw = json.loads((RAW / "recipes" / "recipes.json").read_text(encoding="utf-8"))
catalog_raw = json.loads((RAW / "catalogs" / "products_clean_recipes.json").read_text(encoding="utf-8"))

lines = [i for r in recipes_raw for i in r["ingredients"]]
distinct_raw = {i["name"].strip().lower() for i in lines}
no_amount = sum(i["amount"] is None for i in lines)
print(f"công thức: {len(recipes_raw)} | sản phẩm catalog: {len(catalog_raw)}")
print(f"dòng nguyên liệu thô: {len(lines)} | tên khác nhau: {len(distinct_raw)} | thiếu số lượng: {no_amount/len(lines):.0%}")

công thức: 2527 | sản phẩm catalog: 1129
dòng nguyên liệu thô: 29615 | tên khác nhau: 10311 | thiếu số lượng: 59%


Các kiểu nhiễu quan sát được trong dòng nguyên liệu thô: tiền tố `ĂN KÈM:`, nhiều nguyên liệu trong một dòng, `1M`/`1m` (muỗng canh/cà phê),
phân số `½`, tên thương hiệu Ajinomoto, mô tả sơ chế (băm, bóc vỏ...), cụm `A/B` và `A hoặc B`.

## 2. Bước 1: chuẩn hóa tên nguyên liệu (`scripts/recipe_rag/build_recipes_clean.py`)

**Ý tưởng:** biến mỗi dòng thô thành một hoặc nhiều nguyên liệu dạng `{name, form, quantity, unit, role, alt}`, trong đó `name` là *khóa ghép* với catalog.
Mỗi bước là một hàm nhỏ để giải thích được từng dòng:

| Hàm | Làm gì |
| :-- | :-- |
| `strip_noise` | bỏ `®™*`, dấu ngoặc kép, gọn khoảng trắng |
| `split_prefix` | tách `ĂN KÈM:`/`RAU NÊM:` → `role="side"` |
| `split_head_tail` | tách phần tên và phần số lượng/ghi chú |
| `canonical` | bỏ thương hiệu, bỏ sơ chế, tách `form` dễ làm, bỏ mô tả, áp bảng `ALIASES` |
| `parse_quantity` | đọc số lượng: ưu tiên dữ liệu crawler, rồi `250g`, rồi `1M`, rồi `½ trái` |
| `clean_ingredient` | ghép các bước, tách dòng danh sách, tách `A/B` vào `alt` |

**Quy tắc "đổi bản chất":** dạng khó làm tại nhà là mặt hàng riêng, giữ trong tên; dạng dễ làm thì gộp về tên gốc.

In [3]:
from build_recipes_clean import clean_ingredient

samples = [
    'Hạt nêm Aji-ngon® Heo\t6g (1.5m)',
    'Nước tương "Phú Sĩ" giảm muối',
    'ĂN KÈM: Dưa kiệu, rau thơm, dưa leo cắt sợi',
    'Tương ớt, muối, tiêu',
    'Hành tây:\t1/2 củ (40g)',
    'Thơm: ½ trái',
    'Tôm khô: 20g',
    'Tỏi phi',
    'Ớt chuông đỏ     1/2 trái',
    'Ớt hiểm xanh đỏ',
    'Tôm sú bóc vỏ\t200g',
    'Bột năng pha loãng',
    'Cà hộp',
    'Húng lủi/húng bạc hà',
]
rows = []
for s in samples:
    for r in clean_ingredient({"raw_text": s, "name": s, "amount": None, "unit": None}):
        rows.append([repr(s), r["name"], r["form"] or "", r["quantity"] if r["quantity"] is not None else "",
                     r["unit"] or "", r["role"], ", ".join(r.get("alt", []))])
table(rows, ["dòng thô", "name", "form", "quantity", "unit", "role", "alt"])

| dòng thô | name | form | quantity | unit | role | alt |
| :--| :--| :--| :--| :--| :--| :--|
| 'Hạt nêm Aji-ngon® Heo\t6g (1.5m)' | hạt nêm |  | 6.0 | g | main |  |
| 'Nước tương "Phú Sĩ" giảm muối' | nước tương |  |  |  | main |  |
| 'ĂN KÈM: Dưa kiệu, rau thơm, dưa leo cắt sợi' | dưa kiệu |  |  |  | side |  |
| 'ĂN KÈM: Dưa kiệu, rau thơm, dưa leo cắt sợi' | rau thơm |  |  |  | side |  |
| 'ĂN KÈM: Dưa kiệu, rau thơm, dưa leo cắt sợi' | dưa leo |  |  |  | side |  |
| 'Tương ớt, muối, tiêu' | tương ớt |  |  |  | main |  |
| 'Tương ớt, muối, tiêu' | muối |  |  |  | main |  |
| 'Tương ớt, muối, tiêu' | tiêu |  |  |  | main |  |
| 'Hành tây:\t1/2 củ (40g)' | hành tây |  | 40.0 | g | main |  |
| 'Thơm: ½ trái' | thơm |  | 0.5 | quả | main |  |
| 'Tôm khô: 20g' | tôm khô |  | 20.0 | g | main |  |
| 'Tỏi phi' | tỏi | phi |  |  | main |  |
| 'Ớt chuông đỏ     1/2 trái' | ớt chuông đỏ |  | 0.5 | quả | main |  |
| 'Ớt hiểm xanh đỏ' | ớt hiểm xanh |  |  |  | main |  |
| 'Ớt hiểm xanh đỏ' | ớt hiểm đỏ |  |  |  | main |  |
| 'Tôm sú bóc vỏ\t200g' | tôm sú |  | 200.0 | g | main |  |
| 'Bột năng pha loãng' | bột năng |  |  |  | main |  |
| 'Cà hộp' | cà chua hộp |  |  |  | main |  |
| 'Húng lủi/húng bạc hà' | húng lủi |  |  |  | main | húng bạc hà |

Đọc bảng trên: "tôm khô" giữ nguyên (hàng riêng), "tỏi phi" → `tỏi` + `form=phi` (làm tại nhà được), "ớt hiểm xanh đỏ" tách thành hai mặt hàng,
"bóc vỏ" và "pha loãng" bị bỏ vì là bước sơ chế, "cà hộp" → "cà chua hộp".

In [4]:
clean = json.loads((PROC / "recipes_clean.json").read_text(encoding="utf-8"))
L = [i for r in clean for i in r["ingredients"]]
names = collections.Counter(i["name"] for i in L)
covered = sum(c for c in names.values() if c >= 5) / len(L)

print(f"dòng thô {len(lines)}  →  dòng sạch {len(L)}  (tăng vì dòng danh sách được tách ra)")
print(f"tên khác nhau: {len(distinct_raw)}  →  {len(names)}")
print(f"{sum(1 for c in names.values() if c >= 5)} tên xuất hiện ≥5 lần, phủ {covered:.0%} số dòng  → duyệt tay được")
print("có số lượng:", f"{sum(i['quantity'] is not None for i in L)/len(L):.0%}",
      "| role=side:", sum(i['role'] == 'side' for i in L),
      "| có alt:", sum(1 for i in L if i.get('alt')))
print("form:", dict(collections.Counter(i['form'] for i in L if i['form'])))
print("công thức không còn nguyên liệu nào:", sum(1 for r in clean if not r['ingredients']))
print("top 15:", names.most_common(15))

dòng thô 29615  →  dòng sạch 36464  (tăng vì dòng danh sách được tách ra)


tên khác nhau: 10311  →  3153
650 tên xuất hiện ≥5 lần, phủ 90% số dòng  → duyệt tay được
có số lượng: 44% | role=side: 2151 | có alt: 84
form: {'phi': 300, 'rang': 399, 'xay': 286}
công thức không còn nguyên liệu nào: 5
top 15: [('dầu ăn', 1613), ('hạt nêm', 1498), ('đường', 1457), ('tiêu', 1333), ('tỏi', 1173), ('bột ngọt', 1050), ('muối', 1049), ('hành tím', 889), ('nước tương', 822), ('hành lá', 794), ('nước mắm', 719), ('ớt sừng', 617), ('xốt mayonnaise', 610), ('cà rốt', 596), ('hành tây', 534)]


### Tự đánh giá bước 1

**Đã kiểm chứng (bằng mắt, trên mẫu):** xem ngẫu nhiên ~30 dòng raw → clean hai lần, và đo các nhóm lỗi trước/sau. Kết quả: dấu thừa trong tên 171 → 1, tên dính số lượng → 0,
mảnh "đỏ/vàng" rời 49 → 0, phân số mất số lượng 47 → 4.

**Chưa kiểm chứng:** *không có tập nhãn chuẩn*, nên chưa có số precision/recall thật. Mọi nhận xét "sạch hơn" là nhận xét trên mẫu nhỏ.

**Lỗi và rủi ro đã biết:**
- `PREP_RE` cắt tại chữ *thái*, nên "Xốt thái sả tắc" (có thể là "Thái") thành "xốt".
- Còn mô tả sót lại: "hành lá tỉa hoa", "tỏi đâp giập", "cà chua tán".
- Dòng danh sách có chung dạng ("dưa leo, củ cải và cà rốt ngâm chua ngọt") chỉ gán dạng cho một phần.
- Chỉ nhận ra thương hiệu Ajinomoto, các hãng khác còn nằm trong tên ("xốt mè aji-xốt").
- Bảng `ALIASES` mới khoảng 25 dòng. Các cặp đồng nghĩa còn lại sẽ dựa vào bước 3.
- 5 công thức không còn nguyên liệu nào cần xem lại nguồn.
- Quy tắc "dễ làm / khó làm" là phán đoán của người, chưa có ngoại lệ được duyệt.

**Cách nâng cấp:**
1. Gán nhãn ~200 dòng làm **tập chuẩn + unit test**, để sửa luật mà không sợ hỏng chỗ khác.
2. Đưa bảng luật sang file YAML để cả nhóm sửa mà không đụng code.
3. Dùng **từ vựng catalog làm tín hiệu ngược**: bỏ dần từ đuôi cho tới khi khớp một loại hàng.
4. Phần đuôi dài (6% dòng) có thể để LLM nhỏ chuẩn hóa một lần, rồi lưu cache.

## 3. Bước 2: chuẩn hóa catalog (`scripts/recipe_rag/build_catalog_clean.py`)

**Ý tưởng:** muốn ghép được thì hai phía phải đi qua *cùng một bộ luật*. Script này tái dùng hàm `canonical` của bước 1 và thêm phần riêng cho sản phẩm:

- tách **quy cách** `pack_size`, `pack_unit`, `container` (`gói 200g`, `chai 1 lít`);
- nhận diện **bán nhiều đơn vị** (`18 chai...`, `Thùng 12 chai...`) → `multipack`;
- đoán **hãng** theo cụm chữ viết hoa liền nhau (trường `brand` gốc luôn rỗng);
- `base` = loại hàng đã chuẩn hóa, dùng để ghép với tên nguyên liệu.

In [5]:
from build_catalog_clean import clean_product

rows = []
for p in catalog_raw[:400:40]:
    c = clean_product(p)
    rows.append([c["name"][:50], c["base"], c["form"] or "", c["brand"] or "", c["pack_size"], c["pack_unit"], c["container"] or "", c["multipack"] or ""])
table(rows, ["tên gốc", "base", "form", "brand (đoán)", "pack_size", "pack_unit", "container", "multipack"])

| tên gốc | base | form | brand (đoán) | pack_size | pack_unit | container | multipack |
| :--| :--| :--| :--| :--| :--| :--| :--|
| Nước mắm Nam Ngư Phú Quốc đậm đặc 32 độ đạm chai 5 | nước mắm đậm đặc |  | Nam Ngư Phú Quốc | 500.0 | ml | chai |  |
| Dầu đậu nành Meizan chai 2 lít | dầu đậu nành |  | Meizan | 2.0 | lít | chai |  |
| Bột năng đa dụng Miwon gói 400g | bột năng đa dụng |  | Miwon | 400.0 | g | gói |  |
| Sữa tươi ít đường Dalat Milk 220ml | sữa tươi ít đường |  | Dalat Milk | 220.0 | ml |  |  |
| Kem đặc Vinamilk Tài Lộc lon 380g | kem đặc |  | Vinamilk Tài Lộc | 380.0 | g | lon |  |
| Cháo yến hải sản rong biển Yến Việt 50g | cháo yến hải sản rong biển |  | Yến Việt | 50.0 | g |  |  |
| Mì Kokomi tôm chua cay chanh ớt gói 75g | mì tôm chua cay chanh ớt |  | Kokomi | 75.0 | g | gói |  |
| Mì không cay Jin Ottogi gói 120g | mì không cay |  | Jin Ottogi | 120.0 | g | gói |  |
| Bún giò heo ăn liền Hằng Nga 75g | bún giò heo ăn liền |  | Hằng Nga | 75.0 | g |  |  |
| Chanh không hạt túi 250g (4 trái) | chanh không hạt |  |  | 250.0 | g | túi |  |

In [6]:
cat = json.loads((PROC / "catalog_clean.json").read_text(encoding="utf-8"))
print(f"sản phẩm: {len(catalog_raw)} → {len(cat)} (bỏ trùng tên + quy cách)")
print(f"có pack_size: {sum(c['pack_size'] is not None for c in cat)} | có brand đoán: {sum(c['brand'] is not None for c in cat)} | multipack: {sum(c['multipack'] is not None for c in cat)}")
print(f"loại hàng (base) khác nhau: {len({c['base'] for c in cat})}")
print("sản phẩm không có quy cách:", sum(c['pack_size'] is None for c in cat), "(chủ yếu rau củ, trái cây bán theo cân)")

sản phẩm: 1129 → 1116 (bỏ trùng tên + quy cách)
có pack_size: 990 | có brand đoán: 891 | multipack: 73
loại hàng (base) khác nhau: 895
sản phẩm không có quy cách: 126 (chủ yếu rau củ, trái cây bán theo cân)


### Tự đánh giá bước 2

**Đã kiểm chứng:** xem 22 mẫu ngẫu nhiên, sửa lỗi `base="thùng"` (34 sản phẩm) bằng cách nhận cả dạng "Thùng 12 chai...".
Lỗi thứ hai (`base` còn ngoặc, ví dụ "chanh không hạt (4 )") được phát hiện khi viết notebook này và đã sửa. Điều đó cho thấy 22 mẫu chưa đủ để tin cậy.

**Chưa kiểm chứng:** *độ đúng của `base` trên toàn bộ 1.116 sản phẩm* và độ đúng của `brand`.

**Rủi ro đã biết:**
- `brand` là đoán: "Đức La Cusina" chứa cả chữ "Đức" (nơi sản xuất), "Nam Vang" bị coi là hãng.
- Cùng lỗi chữ *thái* như bước 1, vì dùng chung `canonical`.
- 126 sản phẩm không có quy cách, mà đây lại là rau củ, đúng nhóm cần cho công thức. **Bước quy đổi (mục 6) sẽ phải xử lý, nếu không không tính được số gói.**
- Catalog hiện là dữ liệu crawl từ Bách Hóa Xanh, **chưa có ID Medusa**. Cần một bước ánh xạ khi import.

**Cách nâng cấp:** ghép `brand` với một danh sách hãng thật (lấy từ các hãng xuất hiện ≥3 lần), tách riêng xuất xứ ("Đà Lạt", "Nam Phi"), và với rau củ bán theo cân thì lưu thêm trọng lượng thật khi import vào Medusa.

## 4. Phép đo thô: độ phủ nếu chỉ khớp bằng chữ

Chưa phải ghép chính thức. Cell này chỉ trả lời: *sau chuẩn hóa, bao nhiêu dòng nguyên liệu có loại hàng tương ứng trong catalog?*
Quy tắc: tên nguyên liệu khớp `base` đúng hoặc là tiền tố của `base`, hoặc trùng tên danh mục.

In [7]:
bases = [p["base"] for p in cat]
cats = [p["category"].lower() for p in cat]

def hit(n):
    return any(x == n or x.startswith(n + " ") for x in bases) or any(n == c for c in cats)

total = sum(names.values())
matched = sum(c for n, c in names.items() if hit(n))
print(f"độ phủ theo dòng (khớp chữ thô): {matched/total:.1%}   (trước chuẩn hóa, phép khớp substring thô cho ~30%)")
print("nguyên liệu phổ biến chưa khớp:", [(n, c) for n, c in names.most_common(60) if not hit(n)][:20])

độ phủ theo dòng (khớp chữ thô): 68.7%   (trước chuẩn hóa, phép khớp substring thô cho ~30%)
nguyên liệu phổ biến chưa khớp: [('hành boa rô', 226), ('dầu điều', 212), ('tôm sú', 186), ('ngò gai', 181), ('nước cốt chanh', 171), ('bột tẩm khô chiên giòn', 168), ('rượu trắng', 148), ('đậu phộng', 136), ('nước dừa tươi', 127), ('húng lủi', 111), ('tiêu xanh', 110), ('cơm trắng', 103), ('rau om', 97), ('cần tây', 89)]


### Tự đánh giá mục 4

Độ phủ tăng từ khoảng 30% lên khoảng 69%, cho thấy việc chuẩn hóa **có tác dụng**. Nhưng cần đọc đúng:

- Đây là độ phủ **theo dòng**, không phải theo công thức. Một công thức chỉ dùng được khi **mọi** nguyên liệu `core` của nó khớp, nên tỉ lệ công thức đủ nguyên liệu sẽ thấp hơn nhiều. **Con số này chưa được đo** (bước 4).
- Khớp tiền tố có thể khớp *sai*: "muối" khớp "muối tôm", "tiêu" khớp "tiêu xay". Phép đo này chưa có kiểm tra đúng/sai.
- Các nguyên liệu chưa khớp (hành boa rô, dầu điều, tôm sú, ngò gai...) phần lớn là hàng **catalog chưa bán**, không phải do tên sai. Đây là giới hạn của dữ liệu, không sửa bằng luật được.

**Hiệu chỉnh sau khi có kết quả bge-m3 (mục 5):** con số 69% gộp cả khớp *đúng* và khớp *tiền tố*. Tách riêng thì trong 654 tên phổ biến: chỉ **125 tên khớp đúng**, **103 tên chỉ khớp tiền tố**, và **426 tên (65% số tên, ~25% số dòng) không có khớp chữ nào**.
Khớp tiền tố phần lớn đúng loại hàng ("muối" → "muối hạt", "đường" → "đường kính trắng", "dầu ăn" → "dầu ăn cao cấp"), nhưng có trường hợp sai ("tỏi" → "tỏi giòn" thuộc Gia vị nêm sẵn). Vì vậy 69% là **cận trên lạc quan**, không phải độ phủ thật.

## 5. Bước 3: bge-m3 gợi ý gộp tên và ghép catalog (`scripts/recipe_rag/build_alias_candidates.py`)

**Ý tưởng:** embed (vector 1024 chiều) toàn bộ tên nguyên liệu và `base` của catalog bằng `BAAI/bge-m3`, tính cosine, rồi **đề xuất**:

1. `alias_candidates.csv`: tên hiếm → tên phổ biến giống nó (≥ 0,85), gắn nhãn `variant` hoặc `synonym?`;
2. `catalog_candidates.csv`: mỗi nguyên liệu (≥ 5 lần) → top-3 loại hàng catalog.

**Script không tự gộp.** bge-m3 coi "ớt đỏ" và "ớt xanh" rất giống nhau, mà theo quyết định thì hai thứ này khác nhau. Người duyệt đánh dấu cột `accept`,
dòng đúng được chép vào `ALIASES`.

In [8]:
import csv
ap, cp = PROC / "alias_candidates.csv", PROC / "catalog_candidates.csv"
if ap.exists() and cp.exists():
    alias = list(csv.DictReader(ap.open(encoding="utf-8-sig")))
    cands = list(csv.DictReader(cp.open(encoding="utf-8-sig")))
    syn = [r for r in alias if r["relation"] == "synonym?"]
    print(f"alias_candidates: {len(alias)} dòng ({len(syn)} nhãn synonym?, {len(alias)-len(syn)} variant)")
    table([[r["name"], r["count"], r["merge_into"], r["target_count"], r["similarity"]] for r in syn[:15]],
          ["name", "count", "merge_into", "target_count", "similarity"])
    top1 = [r for r in cands if r["rank"] == "1"]
    lex = sum(1 for r in top1 if r["lexical"])
    print(f"catalog_candidates: {len(top1)} nguyên liệu | top-1 trùng khớp chữ (exact/prefix): {lex}")
    print("Top-1 KHÔNG có khớp chữ (nhóm phổ biến nhất): bge-m3 vẫn trả về một thứ, dù catalog có thể không bán nguyên liệu đó")
    table([[r["name"], r["count"], r["catalog_base"], r["category"], r["similarity"]] for r in top1 if not r["lexical"]][:20],
          ["name", "count", "top-1 catalog base", "category", "similarity"])

    # Phân loại 654 tên theo khớp chữ trên TOÀN catalog (không chỉ top-3)
    cat_bases = [p["base"] for p in cat]
    ex = [r["name"] for r in top1 if r["name"] in cat_bases]
    pre = [r["name"] for r in top1 if r["name"] not in cat_bases and any(b.startswith(r["name"] + " ") for b in cat_bases)]
    non = [r for r in top1 if r["name"] not in ex and r["name"] not in pre]
    tot_lines = sum(int(r["count"]) for r in top1)
    print(f"\n{len(top1)} tên: khớp đúng {len(ex)} | chỉ khớp tiền tố {len(pre)} | không có khớp chữ {len(non)} "
          f"(= {sum(int(r['count']) for r in non)/tot_lines:.0%} số dòng)")
    sims = sorted(float(r["similarity"]) for r in top1 if r["name"] in ex)
    print(f"similarity của top-1 với nguyên liệu CÓ tên trùng đúng trong catalog: min {sims[0]:.2f}, trung vị {sims[len(sims)//2]:.2f}")

    # Cặp gộp tên nào có thể bắt bằng luật đơn giản (đổi thứ tự từ / bỏ dấu), không cần model
    import unicodedata
    def noacc(s): return "".join(c for c in unicodedata.normalize("NFD", s.replace("đ", "d")) if unicodedata.category(c) != "Mn")
    rule = [r for r in syn if sorted(noacc(r["name"]).split()) == sorted(noacc(r["merge_into"]).split())]
    print(f"cặp synonym? bắt được bằng luật đổi-thứ-tự-từ/bỏ-dấu: {len(rule)}/{len(syn)}")
else:
    print("CHƯA CÓ kết quả: alias_candidates.csv / catalog_candidates.csv chưa được tạo.")
    print("Chạy:  .venv\\Scripts\\python scripts\\recipe_rag\\build_alias_candidates.py   rồi chạy lại cell này.")

alias_candidates: 282 dòng (186 nhãn synonym?, 96 variant)


| name | count | merge_into | target_count | similarity |
| :--| :--| :--| :--| :--|
| xà lách romaine | 3 | xà lách romain | 10 | 0.980 |
| bắp hạt mỹ | 3 | bắp mỹ hạt | 13 | 0.975 |
| màu dầu điều | 20 | dầu màu điều | 29 | 0.965 |
| bột ớt paprika | 6 | ớt bột paprika | 19 | 0.965 |
| đậu hủ non | 4 | đậu hũ non | 45 | 0.951 |
| phi lê cá hồi | 4 | cá hồi phi lê | 21 | 0.949 |
| cua thịt | 4 | thịt cua | 26 | 0.945 |
| tàu hủ ky | 8 | tàu hũ ky | 13 | 0.945 |
| tỏi dầu | 4 | dầu tỏi | 13 | 0.944 |
| thịt gà má đùi | 6 | thịt má đùi gà | 8 | 0.944 |
| nước cốt hành tỏi | 5 | nước cốt tỏi | 9 | 0.943 |
| lá húng lủi | 28 | húng lủi | 111 | 0.936 |
| lòng đỏ trứng | 6 | lòng trắng trứng | 11 | 0.935 |
| lá húng quế | 15 | lá quế | 46 | 0.933 |
| thịt ức vịt | 3 | ức vịt | 5 | 0.932 |

catalog_candidates: 654 nguyên liệu | top-1 trùng khớp chữ (exact/prefix): 207
Top-1 KHÔNG có khớp chữ (nhóm phổ biến nhất): bge-m3 vẫn trả về một thứ, dù catalog có thể không bán nguyên liệu đó


| name | count | top-1 catalog base | category | similarity |
| :--| :--| :--| :--| :--|
| tỏi | 1173 | dầu tỏi | Gia vị nêm sẵn | 0.845 |
| trứng gà | 259 | cá trứng | Thủy hải sản, thịt đông | 0.807 |
| hành boa rô | 226 | cá rô | Cá, hải sản | 0.668 |
| dầu điều | 212 | bột điều | Bột nghệ, tỏi, hồi, quế,... | 0.765 |
| tôm sú | 186 | mắm tôm | Nước chấm, mắm | 0.769 |
| ngò gai | 181 | ngò rí | Rau lá | 0.799 |
| nước cốt chanh | 171 | nước cốt dừa | Nước cốt dừa lon | 0.823 |
| bột tẩm khô chiên giòn | 168 | bột khô chiên giòn | Bột các loại | 0.934 |
| thơm | 157 | gạo thơm | Gạo, nếp | 0.811 |
| rượu trắng | 148 | rượu mơ | Rượu | 0.683 |
| đậu phộng | 136 | đậu | Củ, quả | 0.823 |
| nước dừa tươi | 127 | nước màu dừa | Bột nghệ, tỏi, hồi, quế,... | 0.824 |
| húng lủi | 111 | lươn | Cá, hải sản | 0.638 |
| chanh | 111 | bánh canh | Miến, hủ tiếu, phở khô | 0.646 |
| tiêu xanh | 110 | xúc xích vị tiêu xanh | Xúc xích | 0.786 |
| bột mì | 103 | bột năng khoai mì | Bột các loại | 0.816 |
| cơm trắng | 103 | củ cải trắng | Củ, quả | 0.708 |
| rau om | 97 | rau lang | Rau lá | 0.763 |
| cần tây | 89 | măng tây | Củ, quả | 0.749 |
| mè | 89 | mì | Mì ăn liền | 0.846 |


654 tên: khớp đúng 125 | chỉ khớp tiền tố 103 | không có khớp chữ 426 (= 25% số dòng)
similarity của top-1 với nguyên liệu CÓ tên trùng đúng trong catalog: min 0.69, trung vị 1.00
cặp synonym? bắt được bằng luật đổi-thứ-tự-từ/bỏ-dấu: 18/186


### Tự đánh giá bước 3

**Số đo (CPU, máy dev, `BAAI/bge-m3` qua sentence-transformers 6.1.0):** vector **1024 chiều**; embed **3.863 chuỗi trong 261 giây (≈ 68 ms/chuỗi khi embed theo lô)**;
tải và nạp model mất ~2.022 giây, *gần như toàn bộ là tải về lần đầu (~5,5 GB cache)*. Con số 68 ms là theo lô, **không** phải độ trễ của một câu hỏi lẻ lúc chạy thật, phần này chưa đo.

**Kết quả: bge-m3 hữu ích để *gợi ý gộp tên*, nhưng KHÔNG đủ tin cậy để *ghép nguyên liệu → sản phẩm* một mình.**

*Gộp tên đồng nghĩa (186 cặp `synonym?`):* các cặp similarity cao nhất đa số đúng và là lỗi thứ tự từ hoặc chính tả ("bắp hạt mỹ" ↔ "bắp mỹ hạt", "đậu hủ non" ↔ "đậu hũ non", "phi lê cá hồi" ↔ "cá hồi phi lê").
Nhưng **similarity cao vẫn có cặp sai** — "lòng đỏ trứng" → "lòng trắng trứng" (0,935), "rượu vang trắng" → "rượu trắng" (0,923), "hành tây tím" → "hành tím" (0,915), "bắp cải trắng" → "củ cải trắng" (0,874), "tương ớt" → "ớt" (0,852). Không chọn được ngưỡng nào tách đúng/sai. **Bắt buộc duyệt tay.**
Chỉ có 18/186 cặp bắt được bằng luật đơn giản (đổi thứ tự từ, bỏ dấu), phần còn lại cần người quyết định.

*Ghép với catalog (654 nguyên liệu):* chỉ **207** nguyên liệu có top-1 trùng khớp chữ. Với **447** nguyên liệu còn lại bge-m3 *luôn trả về một thứ*, kể cả khi catalog không bán nguyên liệu đó. Trong **30 nguyên liệu phổ biến nhất của nhóm này** (mình đọc tay, chưa gán nhãn), đa số là sai hoặc nguyên liệu vốn không có trong catalog:
"tỏi" → "dầu tỏi" (0,845), "trứng gà" → "cá trứng" (0,807), "tôm sú" → "mắm tôm", "nước cốt chanh" → "nước cốt dừa" (0,823), "húng lủi" → "lươn", "cần tây" → "măng tây", "chanh" → "bánh canh". Một số ít đúng, ví dụ "đầu hành lá" → "hành lá".
Ngược lại, nguyên liệu *có* khớp chữ trong catalog lại có thể có similarity thấp (top-1 khớp chữ thấp nhất chỉ ~0,64). **Điểm similarity không hiệu chỉnh được**, nên một ngưỡng δ duy nhất như proposal mô tả sẽ không tách được "có" và "không có".
Hệ quả thực tế: ghép sai sẽ nạp **"dầu tỏi" vào giỏ khi món cần tỏi** — tệ hơn hẳn việc báo "chưa có trong catalog".

**Đề xuất (chưa làm, cần bàn):** ghép theo thứ tự (1) khớp chữ/tiền tố có kiểm danh mục, (2) bảng ghép **duyệt tay** cho phần còn lại, (3) dùng bge-m3 chỉ để *sinh ứng viên* cho người duyệt, và có danh sách "không có trong catalog" rõ ràng. 654 tên là số lượng duyệt được.

**Chưa kiểm chứng / lưu ý:** `catalog_candidates.csv` được tạo từ `catalog_clean.json` *trước khi* sửa lỗi ngoặc ở bước 2 (ảnh hưởng ít sản phẩm); chưa gán nhãn đúng/sai nên các nhận xét trên là đọc mẫu, không phải precision đo được; các nguyên liệu rất ngắn (như "thơm", "mè", "giá") đặc biệt dễ nhầm ("thơm" → "gạo thơm", "mè" → "mì").

**Cách nâng cấp:** lọc theo `category` trước rồi mới so similarity (docs/05 mục D5); dùng bge-m3 *sparse/lexical weights* hoặc reranker `bge-reranker-v2-m3`;
so với model nhỏ hơn (multilingual-e5, vietnamese-sbert) trên cùng tập nhãn; lưu vector vào pgvector thay vì `.npy` khi dựng `ai_db`.

> **Cập nhật:** mục này là kế hoạch ban đầu. Cuối cùng KHÔNG làm bảng quy đổi đơn vị; thay bằng quy tắc 1 gói và `packs` (mục 9, 15.2).

## 6. Bước 5 (chưa làm): quy đổi đơn vị và quy tắc **mua dư**

Catalog là nguồn sự thật, không đổi đơn vị. Vì vậy công thức phải đổi về đơn vị catalog rồi làm tròn **lên** theo cỡ gói:

```
nhu_cau  = so_luong_trong_cong_thuc × (so_nguoi_yeu_cau / so_nguoi_chuan)     # đơn vị công thức
nhu_cau_g = nhu_cau × gram_moi_don_vi          # bảng quy đổi: 1 trái cà chua ≈ 40g, 1 muỗng canh ≈ 15g...
so_goi    = ceil(nhu_cau_g / pack_size_g)      # luôn ≥ 1 và là số nguyên
```

Cell dưới là **minh họa thiết kế**, *không nằm trong pipeline* và dùng số giả định (40g/trái, gói 500g) để xem cách tính.

In [9]:
import math

def packs_needed(qty, gram_per_unit, pack_g, servings_req=4, servings_std=4):
    # Minh họa: nhu cầu quy ra gam, làm tròn lên số gói (mua dư).
    need_g = qty * gram_per_unit * servings_req / servings_std
    return max(1, math.ceil(need_g / pack_g)), need_g

for label, qty, gpu, pack in [("cà chua: 3 trái, gói 500g", 3, 40, 500),
                              ("cà chua: 3 trái, gói 500g, nấu cho 8 người", 3, 40, 500),
                              ("thịt ba chỉ: 300g, khay 500g", 300, 1, 500),
                              ("nước mắm: 2 muỗng canh (15g), chai 500ml", 2, 15, 500)]:
    n, need = packs_needed(qty, gpu, pack, servings_req=8 if "8 người" in label else 4)
    print(f"{label:48s} cần {need:.0f}g → mua {n} gói")

cà chua: 3 trái, gói 500g                        cần 120g → mua 1 gói
cà chua: 3 trái, gói 500g, nấu cho 8 người       cần 240g → mua 1 gói
thịt ba chỉ: 300g, khay 500g                     cần 300g → mua 1 gói
nước mắm: 2 muỗng canh (15g), chai 500ml         cần 30g → mua 1 gói


### Việc phải làm ở bước này, và rủi ro

- **Bảng quy đổi** (`unit_conversions.json`, có version): trái/quả/củ/cây/nhánh/muỗng → gam, **theo từng nguyên liệu** (1 trái cà chua ≠ 1 trái thơm). Đây là phần cần người viết, không tự động được.
- **Dòng không có số lượng:** ~37% dòng không phải gia vị cơ bản (tỏi, hành tím, hành lá, ngò rí...). Cần quy tắc: gán số lượng mặc định theo nhóm, hay coi là "nêm nếm" và không đưa vào giỏ.
- **Sản phẩm không có quy cách** (rau củ bán theo cân): cần cỡ gói mặc định.
- **Mua dư** là lựa chọn có chủ đích: đảm bảo *đủ*, đổi lại có thể thừa. Nên hiển thị lượng cần so với lượng mua để người dùng hiểu.
- Số gói cần được **tính lại khi khách đổi khẩu phần**, không nên lưu cứng.

## 7. Bước 4: tập công thức mục tiêu (`build_recipe_shortlist.py`, `build_catalog_updates.py`)

**Đổi hướng so với kế hoạch ban đầu:** thay vì ghép cho cả 2.527 món, nhóm chọn **100 món** rồi chỉ làm dữ liệu cho tập đó (proposal yêu cầu ≥ 30 món).
Catalog được *bổ sung sau* nếu thật sự đáng (giữ ưu tiên giới hạn công thức trước).

**Luật khớp (một hàm `find_matches` dùng chung cho cả hai script):**
- khớp khi `base` của sản phẩm (catalog **đầy đủ**, chuẩn hóa cùng luật) trùng đúng hoặc **bắt đầu bằng** tên nguyên liệu; nhận cả dạng "rau + tên" ("Rau húng quế");
- **chặn** danh mục không phải nguyên liệu (snack, bánh kẹo, đồ uống pha sẵn, món làm sẵn, đồ chay...);
- **chặn** hàng chay cho món không chay, và thịt khác loài cho nguyên liệu thịt ("thịt ba chỉ" ≠ "Thịt ba chỉ bò Mỹ"), thịt hộp cho món cần thịt tươi;
- hàng tươi (không có `form`) xếp trước hàng phi/xay/rang.

**Chọn món:** chỉ nguyên liệu `core` quyết định (role `main`, không thuộc `PANTRY`). Xếp theo số core còn thiếu, rồi nguyên liệu thiếu "mở khóa" nhiều món nhất.

In [10]:
short = json.loads((PROC / "recipes_shortlist.json").read_text(encoding="utf-8"))
upd = json.loads((RAW / "catalogs" / "product_clean_recipes_updates_v1.json").read_text(encoding="utf-8"))
status = collections.Counter(r["status"] for r in short)
print("100 món:", dict(status), "| servings mặc định:", [r["name"] for r in short if r["servings_defaulted"]])
src = collections.Counter(i["matches"][0]["source"] for r in short for i in r["ingredients"] if i["kind"] == "core" and i["matches"])
print("nguồn khớp tốt nhất của dòng core:", dict(src))
need = [u for u in upd if u["update_status"] == "need_product"]
print(f"\nproduct_clean_recipes_updates_v1.json: {len(upd)} bản ghi | need_product: {len(need)} | from_full_catalog: {len(upd) - len(need)}")
table([[", ".join(u["ingredients"]), len(u["needed_for"])] for u in sorted(need, key=lambda u: -len(u["needed_for"]))],
      ["nguyên liệu chưa có ở catalog nào", "số món cần"])

100 món: {'full': 49, 'missing_1': 51} | servings mặc định: ['Đậu hủ kho nấm rơm']
nguồn khớp tốt nhất của dòng core: {'exact': 218, 'prefix': 198}

product_clean_recipes_updates_v1.json: 20 bản ghi | need_product: 13 | from_full_catalog: 7


| nguyên liệu chưa có ở catalog nào | số món cần |
| :--| :--|
| thịt ba chỉ | 9 |
| tôm sú | 5 |
| hành boa rô | 4 |
| lá hoành thánh | 4 |
| nước cốt chanh | 4 |
| nước dừa tươi | 4 |
| tiêu xanh | 4 |
| bánh tráng | 3 |
| cá thác lác | 3 |
| cần tây | 3 |
| củ sen | 3 |
| thịt cua | 3 |
| cải bó xôi | 2 |

### Tự đánh giá bước 4

**Sai lầm đã sửa trong lúc làm (ghi lại để không lặp):** lần đo đầu cho "99 món đủ" nhờ luật *chứa cụm từ ở giữa* trên catalog đầy đủ, nhưng luật này khớp sai hàng loạt
("tôm sú" → "Mì 3 Miền tôm sú", "đậu phộng" → "Bơ đậu phộng", "gà ta" → "Trứng gà ta"). Sau khi bỏ luật đó, thêm chặn danh mục, chặn hàng chay/thịt khác loài/thịt hộp và alias cho tên quá chung,
con số thật là **khoảng 49–50 món đủ 100%**. Bài học: *phải đọc các khớp, không tin con số tổng*.

**Còn đáng ngờ, cần duyệt:** "cua" → "Cua đồng xay" (tuỳ món), "cá mòi" → cá mòi hộp (hợp lý cho món bánh mì cá mòi, chưa chắc cho món khác),
`PANTRY` (29 thứ) do AI đoán, `bánh mì`/`nước soda` khớp với hàng có hương vị (matcha, soda kem). Alias "thịt" → "thịt heo", "hành" → "hành lá" là phán đoán theo thói quen công thức Việt.

**Cách nâng cấp:** đưa `PANTRY`, `EXCLUDED_CATEGORIES`, alias ra file cấu hình để nhóm sửa; thêm cột `accept` đã duyệt vào bảng ghép chính thức và dùng nó thay cho luật.

## 8. Bước 5: duyệt tay, bảng ghép chính thức, catalog v2

**Quy trình:** `build_ingredient_review.py` sinh Excel → người duyệt điền → `build_review_v2.py` (AI sửa các chỗ sai, giữ quyết định cũ ở cột `quyet_dinh_goc`)
→ `build_ingredient_matches.py` đọc quyết định, **kiểm tra lỗi** rồi sinh **bảng ghép chính thức** `ingredient_catalog_map_v1.json` → `build_catalog_v2.py` sinh catalog mới.

**Quyết định thiết kế đã chốt trong bước này**
- **Pantry vẫn phải ghép sản phẩm.** Pantry (28 thứ) hiện ở **cuối danh sách, không chọn sẵn**, khách tự bấm chọn thêm vì nhà có thể có hoặc hết. Vì thế `kind = addon`, còn nguyên liệu chính `kind = core` (chọn sẵn).
- **Pantry không cần quy đổi đơn vị:** mỗi món chọn thêm lấy 1 gói. Bảng quy đổi chỉ cần cho nguyên liệu chính.
- **Bỏ hàng thùng/lốc** khỏi ứng viên đầu (khách mua lẻ): sản phẩm `multipack` bị đẩy xuống cuối.
- **Chia pantry thành hai nhóm** (gia vị nền / rau thơm tươi): ghi lại, **chưa làm** (nhóm chốt ưu tiên hoàn thiện trước).
- **Thêm vào catalog:** 19 sản phẩm (16 nguyên liệu chính + 3 pantry) trong `catalog_additions_v1.json`. **Không có giá** (`price = null`), tên/đơn vị/danh mục là giá trị tạm để nhóm điền.

In [11]:
mp = json.loads((REVIEW / "ingredient_catalog_map_v1.json").read_text(encoding="utf-8"))["ingredients"]
print("trạng thái (kind, status):", dict(collections.Counter((o["kind"], o["status"]) for o in mp)))
print("pantry chưa được người duyệt xác nhận:", sum(1 for o in mp if o["kind"] == "addon" and not o["reviewed"]))

v2cat = json.loads((RAW / "catalogs" / "products_clean_recipes_v2.json").read_text(encoding="utf-8"))
print("\ncatalog v2:", len(v2cat), "sản phẩm", dict(collections.Counter(p.get("added_from") or p["source"] for p in v2cat)))

status = {o["ingredient"]: o for o in mp}
short = json.loads((PROC / "recipes_shortlist.json").read_text(encoding="utf-8"))
def needs_new(name):
    o = status[name]; return bool(o["products"]) and o["products"][0]["source"] == "manual_added_v1"
ready = sum(1 for r in short if all(status[i["name"]]["status"] != "not_in_catalog" and not needs_new(i["name"])
                                    for i in r["ingredients"] if i["kind"] == "core"))
after = sum(1 for r in short if all(status[i["name"]]["status"] != "not_in_catalog" for i in r["ingredients"] if i["kind"] == "core"))
print(f"\n100 món: {ready} dùng được ngay với catalog hiện có | {after} sau khi thêm 19 sản phẩm vào catalog")

core = [o for o in mp if o["kind"] == "core" and o["products"]]
nopack = [o for o in core if o["products"][0]["pack_size"] is None]
print(f"nguyên liệu chính có sản phẩm: {len(core)} | sản phẩm đầu tiên KHÔNG có quy cách (pack_size): {len(nopack)}")

trạng thái (kind, status): {('addon', 'matched'): 25, ('addon', 'added'): 3, ('core', 'matched'): 121, ('core', 'added'): 16, ('core', 'alias'): 8}
pantry chưa được người duyệt xác nhận: 28

catalog v2: 1775 sản phẩm {'bachhoaxanh': 1129, 'full_catalog': 2, 'manual_added_v1': 19, 'full_catalog_rest': 625}

100 món: 48 dùng được ngay với catalog hiện có | 100 sau khi thêm 19 sản phẩm vào catalog
nguyên liệu chính có sản phẩm: 145 | sản phẩm đầu tiên KHÔNG có quy cách (pack_size): 42


### Tự đánh giá bước 5

**Kết quả:** 145/145 nguyên liệu chính đã có quyết định; **100/100 món đủ nguyên liệu core sau khi thêm 19 sản phẩm**, và chỉ **~48 món dùng được ngay** với catalog hiện có.
Con số 100/100 là *đúng theo cách xây dựng*: mọi nguyên liệu bị `none` đều được nhóm cam kết thêm vào catalog, nên nó phản ánh kế hoạch, không phải độ phủ thực.

**Đã kiểm chứng:** script đọc quyết định được **thử với 8 kiểu lỗi cố ý** (chọn vượt số ứng viên, alias không tồn tại/vòng/tự trỏ, `sp:` bịa, chữ lạ, ô trống, `ok` mà không có gợi ý, chọn trùng): cả 8 đều bị bắt và không ghi file.
Danh sách 100 món **không đổi** sau khi đổi luật khớp, nên các quyết định đã duyệt vẫn đúng.

**Sai sót của người duyệt mà bảng đã bắt hoặc AI sửa (13 chỗ):** đậu hũ trắng bị khớp đậu hũ chiên, phô mai lát khớp phô mai bào, cua nguyên con khớp cua đồng xay, chanh dây lẫn vào chanh,
cá thu sống khớp cá hộp/cá thu đao, bơ làm bếp khớp trái bơ, nấm bào ngư lẫn nấm khác, ổ bánh mì khớp bánh ngọt, hạt sen tươi/nghệ tươi khớp hàng khô/bột, và một `alias` trỏ về chính nó.
Excel (máy dùng dấu thập phân phẩy) đổi "1,2" thành số 1.2 ở 14 ô: đã ép kiểu chữ. Bài học: *người duyệt cũng sai*, nên cần kiểm tra tự động và đối chiếu với dòng gốc.

**Chưa kiểm chứng / rủi ro:**
- **Pantry chưa được duyệt**: 28 gợi ý do AI chọn. Mọi mục pantry có `reviewed = false`.
- **19 sản phẩm thêm mới là dữ liệu tạm**: không có giá, quy cách và tên chưa phải hàng thật. Không dùng cho báo giá hoặc demo thanh toán trước khi điền.
- **Nhiều sản phẩm không có quy cách:** trong các sản phẩm được chọn đầu tiên, **43/145 không có `pack_size`** (rau củ bán theo cân hoặc theo quả, trứng bán theo hộp 6 quả...). Bước quy đổi và mua dư phải xử lý, nếu không không tính được số gói.
- Alias như "tôm sú" = "tôm thẻ" là quyết định *thay thế loại*; chấp nhận cho MVP nhưng nên ghi rõ với người dùng.

**Cách nâng cấp:** đưa quy cách đếm được vào catalog ("hộp 6 quả" → 6 quả); lưu bảng ghép vào DB thay vì JSON khi nối Medusa; tự động kiểm tra lại bảng mỗi khi catalog đổi.

## 9. Chốt catalog, khóa `catalog_id`, bàn giao cho bên web

**Thay đổi quy tắc số lượng:** bỏ bảng quy đổi đơn vị. Mỗi nguyên liệu = **1 gói theo đơn vị catalog** (mua nguyên gói/lốc). Chỉ khi công thức ghi g/kg/ml/lít mà nhiều hơn quy cách sản phẩm mặc định
thì `packs = làm tròn lên`. Đo trên dữ liệu: chỉ 10/150 dòng so sánh được (7%) cần hơn 1 gói.

**Giá do bên web quản lý.** Catalog bàn giao **không có giá**. Giá crawl không dùng được: nhân 10 so với thực tế (dầu ăn Neptune 1 lít ghi 650.000₫, niêm yết ~65.000₫) và không nhất quán trong cùng catalog.

**Phạm vi catalog:** ban đầu 1.150 sản phẩm (file lọc sẵn + 2 + 19 thêm tay), sau đó **mở rộng thành 1.775** vì catalog này để web hiển thị: thêm 625 sản phẩm thực phẩm còn lại của catalog đầy đủ, loại 10 món không phải đồ ăn (dao thớt, bàn chải, dầu gội). Id cũ giữ nguyên, 625 sản phẩm mới nối `CAT-001151..CAT-001775`. Giỏ hàng công thức chỉ dùng id ≤ `CAT-001150`.

**`catalog_id` khóa cứng** (`scripts/recipe_rag/build_catalog_ids.py`): `CAT-000001..CAT-001775`; khóa nhận diện là `product_url` (duy nhất 100%) hoặc tên nguyên liệu với sản phẩm thêm tay, **không gắn với tên/giá/thứ tự**.
Code thực thi quy tắc: id chỉ được nối thêm, sổ có mã băm và ở chế độ chỉ-đọc, mọi script dừng nếu phát hiện sửa tay.

In [12]:
FINAL = ROOT / "data" / "final"
fcat = json.loads((FINAL / "catalog_v1.json").read_text(encoding="utf-8"))
frec = json.loads((FINAL / "recipes_v1.json").read_text(encoding="utf-8"))
reg = json.loads((REVIEW / "catalog_ids_v1.json").read_text(encoding="utf-8"))
print(f"catalog_v1: {len(fcat)} sản phẩm, id {fcat[0]['catalog_id']}..{fcat[-1]['catalog_id']} | thêm tay: {sum(c['needs_review'] for c in fcat)}")
print("khóa:", reg["locked"], reg["locked_on"], "| sha256", reg["ids_sha256"][:16] + "...")
core_ok = sum(1 for r in frec if all(b["catalog_ids"] for b in r["basket"] if b["kind"] == "core"))
ready = sum(1 for r in frec if not r["depends_on_new_products"])
print(f"công thức đủ nguyên liệu core: {core_ok}/{len(frec)} | dùng được ngay (không cần sản phẩm thêm tay): {ready}/{len(frec)}")
print("giỏ có packs > 1:", [(r["name"][:22], b["ingredient"], b["packs"]) for r in frec for b in r["basket"] if b["packs"] > 1][:4], "...")

catalog_v1: 1775 sản phẩm, id CAT-000001..CAT-001775 | thêm tay: 19
khóa: True 2026-10-21 | sha256 72a9f2ecad73902e...
công thức đủ nguyên liệu core: 100/100 | dùng được ngay (không cần sản phẩm thêm tay): 48/100
giỏ có packs > 1: [('Cách làm bánh mì xíu m', 'thịt heo', 2), ('Bắp non xào mực cay', 'bắp non', 2), ('Cơm Thịt Xào Cay', 'thịt heo', 2), ('Cà tím kho nấm thịt bă', 'thịt heo', 2)] ...


### Tự đánh giá bước chốt

**Đã kiểm chứng:** 0 lỗi kiểm tra tự động; mọi `catalog_id` trong mọi giỏ tồn tại; không giỏ core nào rỗng; chạy lại script gán id lần hai **không đổi id nào**; thử sửa tay một id và xóa một id trong bản sao đều bị chặn.
Phát hiện và sửa khi chốt: 6 tên sản phẩm trùng đang bị bảng ghép chọn *bừa* hàng đầu tiên (ví dụ "Bắp mỹ giòn ngọt" ra gói 3 trái trong khi công thức cần ½ trái); nay chọn tường minh trong `catalog_disambiguation_v1.json`.

**Giới hạn cần nói thật:**
- **"100/100 món đủ nguyên liệu" chỉ đúng nếu 19 sản phẩm thêm tay trở thành hàng thật.** Chỉ **48/100 món** dùng được ngay với hàng đã có.
- 19 sản phẩm thêm tay có tên/đơn vị/danh mục *tạm*, nhóm cần xác nhận. 28 nguyên liệu addon (pantry) do AI gợi ý, chưa được người duyệt.
- Dữ liệu crawl có lỗi gốc không sửa được ở đây, ví dụ sản phẩm "Nấm kim châm Trung Quốc" có url trỏ tới "nấm đùi gà baby 400g". `catalog_id` theo url nên vẫn khóa được, nhưng tên hiển thị sai nằm ở dữ liệu nguồn.
- 2 công thức không có bước làm; 1 công thức không ghi số người (mặc định 4).

**Cách nâng cấp:** đưa sổ id vào DB cùng `catalog_id → medusa variant id`; kiểm tra tự động mỗi khi catalog đổi (sản phẩm mới chỉ nối id, sản phẩm biến mất chuyển `retired`); duyệt nốt pantry.

## 10. Hạ tầng issue #8: `ai_db`, GPU và `requirements`

**`ai_db` (Postgres + pgvector).** Chạy bằng `docker compose` (`docker-compose.yml`), image `pgvector/pgvector:pg16`, container `grocery-ai-db`, cổng **5434** chỉ mở trên `127.0.0.1`
(5432 là `medusa-postgres`, 5433 là `nestfair-postgres`). Tách riêng khỏi `medusa-postgres` vì image đó là `postgres:17` thường, **không có pgvector**, và để không đụng dữ liệu của backend.
Schema ở `db/init/02_schema.sql` gồm 3 bảng: `recipe`, `recipe_ingredient` và `recipe_embedding` (`vector(1024)`, 2 loại chunk `card` / `ingredient`, chỉ mục HNSW cosine).
`ai_db` chỉ giữ công thức, nguyên liệu và embedding; **catalog và giỏ hàng do Medusa quản lý** (ghép theo `catalog_id`).

**GPU.** Máy có RTX 4050 (6 GB) nhưng `torch` ban đầu là bản CPU. Đã cài `torch 2.14.1+cu126` (CUDA 12.6). Ba file `requirements`: `requirements.txt` (service), `requirements-gpu.txt` (bản CUDA), `requirements-data.txt` (script dữ liệu, notebook).

In [13]:
import torch
print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda, "| dùng được:", torch.cuda.is_available())
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print("GPU:", g.name, "|", round(g.total_memory / 2**30, 1), "GB VRAM")

bench = PROC / "bench_embedding_cuda.json"
if bench.exists():
    b = json.loads(bench.read_text(encoding="utf-8"))
    print(f"\nĐo bằng scripts/recipe_rag/bench_embedding.py ({b['model']}, vector {b['dim']} chiều, fp16):")
    table([["tên ngắn (nguyên liệu, loại hàng catalog)", b["names"]["n"], f"{b['names']['seconds']} s", f"{b['names']['ms_each']} ms"],
           [f"thẻ món (~{b['cards']['median_chars']} ký tự)", b["cards"]["n"], f"{b['cards']['seconds']} s", f"{b['cards']['ms_each']} ms"],
           ["1 câu hỏi lẻ", 1, "", f"{b['single_query_ms']} ms"]], ["loại chuỗi", "số lượng", "tổng thời gian", "mỗi chuỗi"])
    print(f"VRAM đỉnh {b['vram_peak_gb']} GB | ước tính embed cả ~39 nghìn đoạn: {b['estimate_all_chunks_s']} giây")
    print(f"fp16 (GPU) so với fp32 (CPU) trên 200 mẫu: cosine thấp nhất {b['fp16_vs_fp32_cosine_min']}, trung bình {b['fp16_vs_fp32_cosine_mean']}")
else:
    print("\nChưa có kết quả đo. Chạy: .venv\\Scripts\\python scripts\\recipe_rag\\bench_embedding.py")
print("So sánh: CPU lần trước (build_alias_candidates.py) mất 261 s cho 3.863 chuỗi ngắn (67,6 ms/chuỗi).")

torch: 2.14.1+cu126 | CUDA build: 12.6 | dùng được: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU | 6.0 GB VRAM

Đo bằng scripts/recipe_rag/bench_embedding.py (BAAI/bge-m3, vector 1024 chiều, fp16):


| loại chuỗi | số lượng | tổng thời gian | mỗi chuỗi |
| :--| :--| :--| :--|
| tên ngắn (nguyên liệu, loại hàng catalog) | 3842 | 1.73 s | 0.45 ms |
| thẻ món (~315 ký tự) | 2527 | 12.68 s | 5.02 ms |
| 1 câu hỏi lẻ | 1 |  | 13.3 ms |

VRAM đỉnh 2.59 GB | ước tính embed cả ~39 nghìn đoạn: 29.1 giây
fp16 (GPU) so với fp32 (CPU) trên 200 mẫu: cosine thấp nhất 0.99975, trung bình 0.99995
So sánh: CPU lần trước (build_alias_candidates.py) mất 261 s cho 3.863 chuỗi ngắn (67,6 ms/chuỗi).


In [14]:
# Kiểm tra ai_db đang chạy (cần Docker bật và file .env). Không in mật khẩu.
try:
    import psycopg
    from dotenv import dotenv_values
    env = dotenv_values(ROOT / ".env")
    with psycopg.connect(host=env["AI_DB_HOST"], port=env["AI_DB_PORT"], dbname=env["AI_DB_NAME"],
                         user=env["AI_DB_USER"], password=env["AI_DB_PASSWORD"], connect_timeout=3) as conn:
        print("kết nối OK:", conn.execute("select current_database(), current_user").fetchone())
        print("pgvector:", conn.execute("select extversion from pg_extension where extname='vector'").fetchone()[0])
        for (t,) in conn.execute("select tablename from pg_tables where schemaname='public' order by 1").fetchall():
            cols = [c for (c,) in conn.execute("select column_name from information_schema.columns where table_name=%s order by ordinal_position", (t,))]
            rows = conn.execute(f'select count(*) from "{t}"').fetchone()[0]
            print(f"  {t}: {rows} dòng | cột: {', '.join(cols)}")
except Exception as ex:
    print("Không kết nối được ai_db:", type(ex).__name__, "- bật Docker và chạy `docker compose up -d`, đảm bảo có .env")

kết nối OK: ('ai_db', 'ai_user')
pgvector: 0.8.7
  recipe: 2527 dòng | cột: id, name, source_url, image_url, description, servings, time_minutes, difficulty, prep_steps, cooking_steps, usage_notes, tips, in_target_set, created_at
  recipe_embedding: 36840 dòng | cột: id, recipe_id, ingredient_id, chunk_type, content, embedding, model, created_at
  recipe_ingredient: 36464 dòng | cột: id, recipe_id, position, raw_text, name, form, quantity, unit, role, alt


### Tự đánh giá hạ tầng

**Đã kiểm chứng:**
- `ai_db` thử ghi và truy vấn vector thật: số chiều 1024, khoảng cách cosine với chính nó bằng 0. Bốn ràng buộc đều chặn đúng dòng sai (chunk `ingredient` thiếu `ingredient_id`, `card` trùng, `role` sai, vector sai số chiều); thử trong giao dịch rồi `ROLLBACK` nên không để lại dữ liệu.
- Python nối được `ai_db` bằng `psycopg` + `pgvector`.
- GPU: số đo ở trên chạy lại được (script nằm trong repo) và **trùng với lần đo đầu**. `pip install --dry-run -r requirements-data.txt` không báo thay đổi nào nên `requirements` khớp `.venv`.

**Sai sót cần ghi lại:**
- Lần thử vector đầu **không chạy thật** (thiếu cờ `-i` của `docker exec`) nhưng in ra `count = 0` trông như đạt; phát hiện và chạy lại.
- Cài `torch` CUDA bằng `pip` **thất bại** (đứt mạng `SSL: UNEXPECTED_EOF`, rồi Windows báo file đang bị giữ), nhưng lệnh báo `exit 0` vì đầu ra bị nối qua `tail`. Cách xử lý: tải bằng `curl` có tải-tiếp, kiểm tra **SHA-256 khớp giá trị chính thức** rồi cài từ file.

**Chưa kiểm chứng / rủi ro:**
- *(Đã giải quyết ở mục 11)* `ai_db` lúc đầu chưa có dữ liệu; `servings` nguồn ghi "0" được đổi thành NULL (69 món chưa rõ khẩu phần).
- **HNSW chưa đo hiệu quả**, và schema ban đầu có **lỗi chỉ mục** (chỉ mục chung cho 2 loại chunk làm tìm kiếm ra 0 dòng), đã sửa, xem mục 11.
- Độ giống fp16/fp32 mới đo trên 200 mẫu, chưa đo ảnh hưởng lên thứ hạng kết quả tìm kiếm.
- Số đo 1 câu hỏi lẻ (13 ms) là trên GPU; **chưa đo trên CPU**, nên nếu server không có GPU thì ngân sách 80 ms chưa được kiểm chứng.

**Cách nâng cấp:** đưa schema vào công cụ migration thay vì chỉ chạy lúc khởi tạo; thêm `Dockerfile` cho service để ghép vào Docker Compose chung.

### Đối chiếu tiêu chí hoàn thành issue #8

| Tiêu chí | Trạng thái | Ghi chú |
| :-- | :-: | :-- |
| Bộ công thức đã chuẩn hóa, có mô tả cấu trúc | ✅ gần xong | `recipes_clean.json`, `recipes_v1.json`; cấu trúc mô tả trong notebook và `data/recipe_rag/final/README.md` |
| ai-service chạy được, `/health` OK, nối được `ai_db` | ⬜ | `ai_db` đã có và nối được từ Python; **chưa có FastAPI**: `src/main.py` và `health.py` vẫn rỗng |
| Đã embed toàn bộ công thức vào pgvector | ✅ | 2.527 thẻ món + 34.313 dòng nguyên liệu (xem mục 11) |
| Kết quả thử truy vấn, nhận xét | ✅ | đã thử, nhận xét ở mục 11; **mới đọc bằng mắt**, chưa phải Recall@5 |
| Hướng dẫn chạy trong README của ai-service | ⬜ | README hiện là mô tả kiến trúc NestJS cũ |

### Kế hoạch embed và hướng GraphRAG (ghi nhận, **chưa thử**)

**Giả thuyết cách embed** (cần đo trên 50–100 câu hỏi tự gán nhãn, Recall@5, so với tìm bằng chữ ILIKE):
1. Chunk `card` = *thẻ ngắn* (tên món, mô tả, vài nguyên liệu chính, **thêm cách viết không dấu**), không nhét cả cách làm vì làm loãng nghĩa; câu hỏi thường là tên món hoặc nguyên liệu.
2. Chunk `ingredient` cho câu hỏi theo nguyên liệu; điểm của món = điểm cao nhất trong các nguyên liệu khớp.
3. Tách số người ("cho 4 người") khỏi câu hỏi trước khi embed.
4. So 3 cách chia: chỉ `card`, chỉ `ingredient`, cả hai.

**GraphRAG:** *chưa cần cho MVP.* Dữ liệu đã có sẵn dạng đồ thị (bảng `recipe_ingredient` nối món với nguyên liệu), nên câu hỏi nhiều bước như gộp nguyên liệu nhiều món (mục C của `docs/05`, chưa nằm trong cam kết)
có thể làm bằng SQL kết bảng. GraphRAG đầy đủ (AI dựng đồ thị rồi tóm tắt cụm) tốn kém, hợp với kho văn bản rối. Nếu đưa vào báo cáo thì nên ghi là hướng phát triển, vì chưa có số đo nào hỗ trợ.

## 11. Nạp dữ liệu, embed và tìm kiếm (3 script, chạy tay)

Chạy từ thư mục `grocery-store-ai` (Docker đã bật, có `.env`):

```powershell
docker compose up -d
.venv\Scripts\python scripts\recipe_rag\load_recipes.py --reset                # nạp 2.527 công thức + nguyên liệu
.venv\Scripts\python scripts\recipe_rag\embed_recipes.py                       # embed 2 loại chunk, tự dùng GPU
.venv\Scripts\python scripts\recipe_rag\search_recipes.py "canh chua"          # thử truy vấn (thêm --mode ingredient, -k 10, --target-only)
```

| Script | Việc làm |
| :-- | :-- |
| `scripts/recipe_rag/ai_db.py` | kết nối `ai_db` từ `.env` (dùng chung) |
| `scripts/recipe_rag/load_recipes.py` | nạp `recipes_clean.json` vào `recipe`, `recipe_ingredient`; `servings` "0" → NULL; đánh dấu 100 món `in_target_set`; `--reset`, `--limit`, `--dry-run` |
| `scripts/recipe_rag/embed_recipes.py` | `card` = tên + bản không dấu + mô tả + nguyên liệu chính; `ingredient` = tên nguyên liệu chính (+ dạng phi/xay/rang); chuỗi trùng chỉ embed **một lần**; chạy lại không nhân đôi (upsert) |
| `scripts/recipe_rag/search_recipes.py` | tách "cho N người" khỏi câu hỏi, embed câu hỏi, lấy top-k theo cosine |

**Lần chạy thật (bạn chạy bằng tay, 2026-10-22):** nạp 2.527 công thức, 36.464 dòng nguyên liệu, 100 món mục tiêu, 69 món chưa rõ khẩu phần. Embed: nạp model 5,7 s; 2.527 thẻ món 18,7 s;
34.313 dòng nguyên liệu chỉ có **3.032 chuỗi khác nhau** nên embed 1,8 s; tổng 58,8 s (gồm ghi vào DB). Kết quả: `card` = 2.527, `ingredient` = 34.313.

In [15]:
# Dữ liệu đang có trong ai_db (cần Docker bật và .env)
try:
    import psycopg
    from dotenv import dotenv_values
    env = dotenv_values(ROOT / ".env")
    with psycopg.connect(host=env["AI_DB_HOST"], port=env["AI_DB_PORT"], dbname=env["AI_DB_NAME"],
                         user=env["AI_DB_USER"], password=env["AI_DB_PASSWORD"], connect_timeout=3) as conn:
        print("recipe:", conn.execute("select count(*) from recipe").fetchone()[0],
              "| thuộc 100 món mục tiêu:", conn.execute("select count(*) from recipe where in_target_set").fetchone()[0],
              "| chưa rõ khẩu phần:", conn.execute("select count(*) from recipe where servings is null").fetchone()[0])
        print("recipe_ingredient:", conn.execute("select count(*) from recipe_ingredient").fetchone()[0])
        print("recipe_embedding:", dict(conn.execute("select chunk_type, count(*) from recipe_embedding group by 1 order by 1").fetchall()))
        print("chỉ mục HNSW:", [r[0] for r in conn.execute("select indexname from pg_indexes where tablename='recipe_embedding' and indexname like '%hnsw%' order by 1")])
        HAS_DATA = conn.execute("select count(*) from recipe_embedding").fetchone()[0] > 0
except Exception as ex:
    HAS_DATA = False
    print("Không kết nối được ai_db:", type(ex).__name__)

recipe: 2527 | thuộc 100 món mục tiêu: 100 | chưa rõ khẩu phần: 69
recipe_ingredient: 36464
recipe_embedding: {'card': 2527, 'ingredient': 34313}
chỉ mục HNSW: ['recipe_embedding_card_hnsw_idx', 'recipe_embedding_ingredient_hnsw_idx']


In [16]:
# Chạy chính script tìm kiếm (không sao chép code), mỗi câu hỏi nạp model nên mất vài giây
import subprocess, os
def search(query, *extra):
    r = subprocess.run([sys.executable, str(ROOT / "scripts" / "search_recipes.py"), query, "-k", "3", *extra],
                       capture_output=True, text=True, encoding="utf-8", cwd=ROOT, env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    return [l for l in r.stdout.splitlines() if l.strip()]

if HAS_DATA:
    for q, extra in [("canh chua", ()), ("món nướng cho 4 người", ()), ("canh chua ca loc", ()), ("canh chua cá lóc", ()),
                     ("xe máy honda", ()), ("canh chua", ("--target-only",))]:
        print(f"\n### {q} {' '.join(extra)}")
        print("\n".join(search(q, *extra)))
else:
    print("Chưa có dữ liệu trong ai_db: chạy load_recipes.py và embed_recipes.py trước.")


### canh chua 


câu hỏi: 'canh chua' | embed 524 ms
1. 0.633  Canh chua Thái  [canh-chua-thai] | 4 người
2. 0.632  Canh chua bông so đũa chay  [canh-chua-bong-so-dua-chay] | 4 người
3. 0.622  Canh chua cá kèo bông so đũa  [canh-chua-ca-keo-bong-so-dua] | 4 người

### món nướng cho 4 người 


câu hỏi: 'món nướng' | khẩu phần yêu cầu: 4 người | embed 503 ms
1. 0.640  Nem nướng giòn rụm, thơm ngon, ăn kèm rau sống  [nem-nuong] | ? người
2. 0.623  Thịt gà nướng  [thit-ga-nuong] | 4 người
3. 0.621  Thịt nướng BBQ  [thit-nuong-bbq] | ? người

### canh chua ca loc 


câu hỏi: 'canh chua ca loc' | embed 560 ms
1. 0.561  Canh chua cá lóc  [canh-chua-ca-loc] | 4 người
2. 0.553  Canh chua cá lóc đậu tương  [canh-chua-ca-loc-dau-tuong] | 4 người
3. 0.552  Canh cá thác lác nấu hoa  [canh-ca-thac-lac-nau-hoa] | 4 người

### canh chua cá lóc 


câu hỏi: 'canh chua cá lóc' | embed 441 ms
1. 0.709  Canh chua cá lóc  [canh-chua-ca-loc] | 4 người
2. 0.694  Canh chua cá lóc đậu tương  [canh-chua-ca-loc-dau-tuong] | 4 người
3. 0.648  Canh chua cá  [canh-chua-ca] | 4 người

### xe máy honda 


câu hỏi: 'xe máy honda' | embed 511 ms
1. 0.424  Heo quay hon  [heo-quay-hon] | ? người
2. 0.411  Gyoza cá hồi - Há cảo Nhật Bản  [gyoza-ca-hoi-ha-cao-nhat-ban] | 4 người
3. 0.407  Bánh xèo Nhật Bản Okonomiyaki chuẩn vị dễ làm  [banh-xeo-nhat-ban-2] | 4 người

### canh chua --target-only


câu hỏi: 'canh chua' | embed 464 ms
1. 0.511  Canh hến nấu bầu  [canh-hen-nau-bau] ★100 món | 4 người
2. 0.499  Canh cá chim nấu ngót  [canh-ca-chim-nau-ngot] ★100 món | 4 người
3. 0.488  Ếch xào cải chua  [ech-xao-cai-chua] ★100 món | 4 người


### Lỗi chỉ mục khiến tìm kiếm ra 0 dòng (đã sửa)

Lần chạy đầu của `search_recipes.py` **không in ra kết quả nào** (cũng không báo lỗi). Nguyên nhân:
- Schema ban đầu có **một chỉ mục HNSW chung** cho cả `card` và `ingredient`. HNSW chỉ lấy khoảng 40 vector gần nhất (`hnsw.ef_search`) **trên toàn bảng** rồi mới áp bộ lọc `chunk_type = 'card'`.
- Câu hỏi ngắn ("canh chua") giống các *tên nguyên liệu ngắn* hơn thẻ món, nên cả 40 vector gần nhất đều là nguyên liệu và bị lọc hết.
- Chứng minh: cùng truy vấn, **tắt chỉ mục thì ra đúng** ("Canh chua Thái" 0,633...); trong phiên Python, kế hoạch thực thi dùng `Index Scan using recipe_embedding_hnsw_idx` kèm `Filter`.
- Hướng ngược lại cũng hỏng: tìm nguyên liệu bằng vector của một thẻ món trả về 0 dòng.

**Cách sửa:** hai chỉ mục HNSW **từng phần**, mỗi loại chunk một cái (`... WHERE chunk_type = 'card'` và `'ingredient'`), `db/migrations/001_partial_hnsw_per_chunk_type.sql` (đã áp dụng, 9 s) và cập nhật `db/init/02_schema.sql`.
`search_recipes.py` giờ **in cảnh báo rõ ràng khi không có kết quả**, để thất bại không còn im lặng.

**Sai sót khi chẩn đoán:** phép thử đầu của tôi nối với bảng tạm nên bộ lập kế hoạch không dùng HNSW và *không tái hiện được lỗi*; chỉ khi chạy đúng truy vấn trong Python mới thấy.
Ban đầu người dùng cũng hiểu nhầm là "RAG kém", trong khi đó là lỗi hạ tầng, không phải chất lượng tìm kiếm.

### Kết quả tìm kiếm (đọc bằng mắt, **chưa phải Recall@5**)

| Câu hỏi | Top 1 | Nhận xét |
| :-- | :-- | :-- |
| "canh chua" | Canh chua Thái (0,633) | cả 5 kết quả đầu đều là canh chua |
| "món nướng cho 4 người" | Nem nướng (0,640) | cả 5 đều món nướng; "4 người" được tách ra |
| "canh chua cá lóc" | Canh chua cá lóc (0,709) | đúng; hạng 2 "cá lóc đậu tương" (0,694) |
| "canh chua ca loc" (không dấu) | Canh chua cá lóc (**0,561**) | tìm trúng top 1 nhưng điểm thấp hơn nhiều, hạng 3 đã lệch |
| `--mode ingredient` "gà" | nhiều món cùng điểm 1,000 | khớp đúng chữ nên **hòa điểm**, thứ tự giữa chúng không có ý nghĩa |
| *(không liên quan)* "xe máy honda" | Heo quay hon (**0,424**) | điểm của kết quả sai |
| *(không liên quan)* "lập trình python" | Lưu ngay công thức... (**0,385**) | |
| *(không liên quan)* "thời tiết hôm nay" | Gỏi mực cay (**0,460**) | |

Câu đúng cho 0,56–0,71, câu sai 0,38–0,46: có khoảng cách nhưng **không lớn** (câu không dấu đúng nhất 0,561 chỉ hơn câu sai nhất 0,460 khoảng 0,10). Ngưỡng ~0,5 chỉ là điểm xuất phát, **cần bộ câu hỏi có nhãn mới chọn được**.

### Phát hiện quan trọng: 100 món có giỏ hàng **không phải món phổ biến**

Trong mọi câu hỏi thử ở trên, **không kết quả nào thuộc 100 món có giỏ hàng** (không có dấu ★). Khi giới hạn `--target-only` thì kết quả kém hẳn:
"canh chua" → *Canh hến nấu bầu* (0,511); "canh chua cá lóc" → *Canh cá chim nấu ngót* (0,547); "gà chiên" → *Chả nấm chiên giòn* (0,566); "bò xào" → *Cơm thịt xào cay* (0,515). Điểm này gần mức của câu không liên quan.
Lý do: 100 món được chọn theo **món nào đủ nguyên liệu trong catalog**, không theo độ phổ biến. Trong 100 món chỉ có 9 món "canh", 6 món "nướng", 10 món "gà", và không có món canh chua nào.

Các món phổ biến **thiếu 2–9 nguyên liệu** (chưa tính 19 sản phẩm thêm tay), nhiều cái là tên gọi khác của hàng đã có:

| Món | Thiếu |
| :-- | :-- |
| Canh chua cá lóc | bạc hà, ngò gai, rau om |
| Canh chua sườn | ngò gai, ngò om, tỏi dầu |
| Thịt gà nướng | dầu màu điều (= `dầu điều`, đã có trong pantry) |
| Thăn bò nướng | gừng mài, thăn bò |
| Canh chua Thái | cốt chanh, cốt dừa, hành tím củ, lá chúc, riềng, tôm càng lóng |
| Thịt nướng BBQ | 8 thứ (bột ớt, cà chua hộp, lá mè, mù tạt vàng, nạm bò úc, thịt ba chỉ...) |

Vì vậy mở rộng giỏ hàng là khả thi, nhưng cần **chọn tập món phổ biến để hỗ trợ**, bổ sung gộp tên gọi khác và thêm sản phẩm vào catalog (id nối tiếp từ `CAT-001776`, không đụng id đã khóa).

### Tự đánh giá mục 11

**Đã kiểm chứng:** dữ liệu thật đã nạp và embed, số đếm khớp (2.527 / 36.464 / 2.527 thẻ / 34.313 nguyên liệu); lỗi chỉ mục được tái hiện, chứng minh và sửa; tìm kiếm cho đúng món ở các câu hỏi thử.

**Chưa kiểm chứng:** *Recall@5* (chưa có bộ câu hỏi có nhãn), so sánh với tìm bằng chữ (giả thuyết H5), hiệu quả thật của HNSW so với quét chính xác (thẻ món chỉ 2.527 nên quét chính xác có thể đã đủ), tác dụng của bản không dấu trong thẻ (mới thấy điểm thấp hơn), ngưỡng "không tìm thấy".

**Hạn chế đã biết:** chế độ `ingredient` bị hòa điểm; 69 món chưa rõ khẩu phần; 2 công thức không có bước làm; kết quả đọc bằng mắt trên vài câu hỏi nên dễ thiên lệch.

> **Đã xử lý ở mục 12:** giỏ hàng nay phủ 1.061 món phổ biến.


## 12. Mở rộng catalog cho các món phổ biến (đợt 2)

**Vì sao:** 100 món có giỏ hàng chọn theo độ phủ catalog nên *không phải món phổ biến* (mục 11). Muốn khách hỏi "canh chua", "gà nướng" thì ra món có giỏ, phải thêm sản phẩm vào catalog.

**Định nghĩa món phổ biến** (`scripts/recipe_rag/suggest_catalog_expansion.py`): làm sạch tên công thức, đếm cụm 2 từ như "canh chua", "gà nướng"; cụm có từ chỉ món (canh, kho, xào…) lặp lại ≥ 6 lần là một **họ món** (thêm vài họ nền tảng như bánh xèo, chả giò). Kết quả: **106 họ món, 1.015 công thức**. Không cần hỗ trợ mọi món của họ: xếp món theo ít thiếu nguyên liệu nhất, hỗ trợ K món đầu mỗi họ. Cuối cùng chọn phủ **cả 1.015**.

**Quy trình đã làm**

| Bước | Script / file | Kết quả |
| :-- | :-- | :-- |
| 1. Nguyên liệu thiếu + gợi ý gộp tên (bge-m3) | `suggest_catalog_expansion.py` → `catalog_expansion_review_v1.xlsx` | 358 gộp tên do người duyệt chấp nhận, 33 không gộp |
| 2. Tính lại ứng viên theo quyết định gộp | `triage_catalog_expansion.py candidates` | 814 tên nguyên liệu cần xét (cho cả 1.015 món) |
| 3. AI duyệt sơ bộ, chia tầng cho người | `triage_catalog_expansion.py apply` → `catalog_expansion_triage_v1.xlsx` | tầng: cần bạn quyết 23, kiểm nhanh 194, hàng hiếm 199, AI chắc chắn 398 |
| 4. Người duyệt, đồng ý theo AI | `catalog_expansion_ai_decisions_v1.json` (3 sửa tay: tai vị, xá bấu, ớt ba tri) | hàng hiếm cũng được thêm |
| 5. Gom thành danh sách sản phẩm | `build_catalog_additions_v2.py` | 394 sản phẩm mới, 197 tên gộp/làm từ nguyên liệu có sẵn, 53 tên bỏ |
| 6. **Dò trùng** | `check_catalog_duplicates.py`, `dedup_catalog_additions_v2.py` | 51 nghi trùng: 22 trùng thật với sản phẩm cũ (không thêm), 18 trùng nhau (gộp), 10 giữ → **354 sản phẩm mới** |
| 7. Thêm vào catalog, gán id | `build_catalog_v2.py`, `build_catalog_ids.py` | **CAT-001776..CAT-002129**; 1.775 id cũ không đổi (đã đối chiếu) |
| 8. Bàn giao v2 | `build_final_handoff_v2.py` → `data/recipe_rag/final/*_v2.*` | catalog 2.129 sản phẩm, **1.061 công thức, 1.061 đủ nguyên liệu chính** |

**Món nhỏ làm từ nguyên liệu khác** (nước me, mỡ hành, vỏ chanh mài, lòng trắng trứng, caramel…) không thêm sản phẩm. Quyết định `che_bien_tu` trỏ về nguyên liệu gốc (me, hành lá, chanh, trứng…). Giỏ hàng chỉ cần có nguyên liệu gốc.

**Thứ tự tìm sản phẩm cho một tên nguyên liệu** (`build_final_handoff_v2.py`, dừng ở bước đầu có kết quả): sản phẩm mới/đích trùng → tên sản phẩm chính xác → bảng ghép 100 món → quyết định bỏ/gộp/làm-từ → gộp tên đã duyệt → luật khớp tên → (không khớp và là cách sơ chế/món tự nấu thì bỏ khỏi giỏ).

### Tự đánh giá mục 12

- **Chắc chắn:** id cũ không đổi, không `catalog_id` trùng, mọi nguyên liệu đã ghép trỏ tới sản phẩm có trong catalog (kiểm tự động, 0 lỗi).
- **Chưa chắc:** 354 quyết định thêm/gộp là **phán đoán của AI**, người duyệt đồng ý nhưng chỉ xem tầng cần quyết và kiểm nhanh. Tên, đơn vị, danh mục của sản phẩm mới là giá trị tạm (373 sản phẩm thêm tay, kể cả 19 của đợt 1, chưa có giá và ảnh).
- **"Đủ nguyên liệu" chỉ nghĩa là có sản phẩm cho mọi nguyên liệu chính**, chưa nghĩa là giỏ đúng. Sản phẩm mặc định là kết quả khớp đầu tiên (ví dụ "thịt đầu vai" ra thịt heo xay). Mục 13 đo chất lượng này.
- Lỗi phát hiện trên đường: `not_purchasable` loại nhầm tên sản phẩm dài (> 6 từ) và tên 2 ký tự (hẹ, lê, gà); đã đổi thứ tự thử khớp tên trước khi bỏ.

## 13. Thử RAG đầu-cuối: câu hỏi → món → giỏ hàng (`scripts/recipe_rag/test_rag_basket.py`)

Chạy: `.venv\Scripts\python scripts\recipe_rag\test_rag_basket.py` (cả bộ câu hỏi) hoặc `... test_rag_basket.py "gà kho gừng cho 4 người"` (một câu, in giỏ hàng). Cần ai_db đang chạy. Kết quả ghi ở `data/recipe_rag/review/rag_test_results.json`.

Ba phần: **A** tìm món bằng thẻ món (bge-m3, cosine) chỉ trong 1.061 món có giỏ hàng; **B** in giỏ hàng của món top-1; **C** đo độ khớp *tên nguyên liệu ↔ tên sản phẩm catalog* trên toàn bộ giỏ hàng.

In [ ]:
# Thử một câu hỏi (in top-5 món và giỏ hàng của món đầu)
# .venv\Scripts\python scripts\recipe_rag\test_rag_basket.py "canh chua cá lóc cho 4 người"

# Chạy cả bộ 28 câu hỏi và đo độ khớp toàn bộ giỏ hàng (ghi data/recipe_rag/review/rag_test_results.json)
# .venv\Scripts\python scripts\recipe_rag\test_rag_basket.py

### A. Tìm món

Bộ thử gồm 27 câu hỏi có từ khóa món mong đợi (không dấu, mọi từ khóa phải có trong tên món) và 1 câu ngoài chủ đề.

**Top-1 đúng 22/27, món đúng nằm trong top-5 (Recall@5) 23/27.**

| Câu hỏi | Món top-1 | sim | Kết quả |
| :-- | :-- | :-: | :-- |
| canh chua cá lóc | Canh chua cá lóc | 0.71 | ĐÚNG |
| gà kho gừng | Nấm đùi gà kho gừng | 0.63 | ĐÚNG |
| sườn xào chua ngọt | Sườn xào chua ngọt | 0.75 | ĐÚNG |
| bò xào hành tây | Khoai tây xào thịt bò | 0.66 | SAI (không có món này trong dữ liệu; ra món gần nhất) |
| cá kho tộ | Cá rô kho tộ | 0.64 | ĐÚNG |
| lẩu thái hải sản | Lẩu gà Thái Lan | 0.60 | ĐÚNG |
| bún riêu cua | Bún riêu cua bắp bò | 0.67 | ĐÚNG |
| thịt kho trứng | Khổ qua xào trứng thịt bằm | 0.61 | SAI (không có món 'thịt kho trứng'; ra món gần nhất) |
| bánh xèo | Bánh xèo cuốn áp chảo | 0.65 | ĐÚNG |
| chả giò | Chả giò sứa | 0.63 | ĐÚNG |
| canh bí đao nấu tôm | Canh Bí Đao Rong Biển | 0.70 | ĐÚNG |
| rau muống xào tỏi | Rau muống xào mắm ruốc | 0.68 | ĐÚNG |
| đậu hũ sốt cà chua | Cà chua nhồi đậu hũ | 0.61 | ĐÚNG |
| cơm chiên dương châu | Cơm chiên Dương Châu thơm ngon dễ làm tại nhà | 0.67 | ĐÚNG |
| mực xào sa tế | Mực xào xốt tiêu xanh | 0.62 | SAI (không có; ra mực xào xốt tiêu xanh / sả ớt) |
| tôm rang me | Mì Trộn Vị Thái | 0.51 | SAI (không có món này trong dữ liệu; ra món gần nhất) |
| gà nướng mật ong | Thịt gà nướng | 0.64 | ĐÚNG |
| mì ý sốt bò bằm | Mì Ý bò băm | 0.66 | ĐÚNG |
| cà ri gà | Gà xào cà ri | 0.64 | ĐÚNG |
| trứng chiên | Gỏi trứng chiên | 0.68 | ĐÚNG |
| canh khổ qua nhồi thịt | Canh khổ qua hầm thịt | 0.70 | ĐÚNG |
| gỏi cuốn tôm thịt | Gỏi cuốn tôm chiên | 0.68 | ĐÚNG |
| ga kho gung cho 4 nguoi | Ốc bươu kho gừng | 0.40 | đúng ở top-5 (chỉ có 'Nấm đùi gà kho gừng'; gõ không dấu thì tụt xuống top-5) |
| món nướng cho 4 người | Thịt gà nướng | 0.62 | ĐÚNG |
| cháo gà | Đậu bắp thịt gà xào chao | 0.61 | ĐÚNG |
| lươn xào sả ớt | Lươn kho sả ớt | 0.67 | ĐÚNG |
| canh chua bồn bồn | Canh chua bồn bồn vị thái | 0.72 | ĐÚNG |
| điện thoại iphone 15 | Cơm chiên thập cẩm | 0.35 | ngoài chủ đề (ngoài chủ đề: điểm thấp nhất) |

**Đọc kết quả**

- 4 trong 5 câu không đứng đầu là do **món đó không có trong dữ liệu** (đã tra tên 1.061 món), không phải lỗi tìm kiếm: hệ thống trả món gần nhất (`bò xào hành tây` → khoai tây xào thịt bò). Tập câu hỏi do mình tự đoán, không phải truy vấn thật của khách.
- Câu thứ 5: gõ **không dấu** (`ga kho gung`) làm món đúng tụt từ top-1 xuống top-5, trong khi gõ có dấu thì đúng top-1. Thẻ món đã nhúng thêm bản không dấu nhưng chưa đủ; cần thêm câu hỏi không dấu vào bộ thử.
- **Không tách được ngưỡng "không tìm thấy" sạch:** câu đúng top-1 có sim từ 0.60 (trung vị 0.67); câu sai 0.40–0.66; câu ngoài chủ đề 0.35. Ngưỡng ~0.55 loại được iphone nhưng cũng sẽ nhận món gần-đúng như "bò xào hành tây" (0.66). Cần nhãn thật để chọn ngưỡng, hoặc dùng kèm điều kiện "tên món chứa từ khóa".
- Số liệu này **chưa phải Recall@5 chính thức của issue #8**: 27 câu, nhãn tự soạn, chưa có baseline `ILIKE` để so (H5).

### B. Giỏ hàng của món đúng

Ví dụ `canh chua cá lóc` (top-1, sim 0.71): mua nguyên gói, `packs` = 1.

| Loại | Nguyên liệu | Sản phẩm catalog mặc định | sim | cách ghép |
| :-- | :-- | :-- | :-: | :-- |
| core | bạc hà | Bạc hà (dọc mùng) 300g | 0.61 | sản phẩm mới đợt 2 |
| core | cà chua | Cà chua 300g | 0.74 | bảng ghép 100 món (người duyệt) |
| core | cá lóc | Cá lóc làm sạch vỉ 300g | 0.65 | luật khớp tên |
| core | giá | Giá mầm sạch 300g | 0.56 | luật khớp tên |
| core | me | Me chua vắt gói 250g | 0.43 | bảng ghép 100 món (người duyệt) |
| core | ngò gai | Ngò gai, rau om 50g | 0.66 | AI quyết định gộp |
| core | rau om | Ngò gai, rau om 50g | 0.61 | AI quyết định gộp |
| core | thơm | Thơm trái trung từ 700g - dưới 1kg | 0.45 | bảng ghép 100 món (người duyệt) |
| core | đậu bắp | Đậu bắp khay 250g | 0.64 | bảng ghép 100 món (người duyệt) |
| core | ớt sừng | Ớt sừng đỏ tươi gói 100g | 0.71 | bảng ghép 100 món (người duyệt) |
| addon | bột ngọt | Bột ngọt Meizan gói 400g | 0.58 | bảng ghép 100 món (người duyệt) |
| addon | dầu ăn | Dầu ăn thượng hạng Neptune 1 lít | 0.57 | bảng ghép 100 món (người duyệt) |
| addon | muối | Muối tinh sấy iot Sosal Group 1kg | 0.54 | bảng ghép 100 món (người duyệt) |
| addon | nước mắm | Nước mắm Phú Quốc Thanh Quốc 520ml | 0.57 | bảng ghép 100 món (người duyệt) |
| addon | sả | Sả cây gói 200g | 0.64 | bảng ghép 100 món (người duyệt) |
| addon | tỏi | Tỏi tím Trung Quốc 200g | 0.58 | bảng ghép 100 món (người duyệt) |
| addon | đường | Đường kính trắng An Khê 500g | 0.44 | bảng ghép 100 món (người duyệt) |

Cột `sim` là độ giống bge-m3 giữa *tên nguyên liệu* và *tên sản phẩm* (tên sản phẩm có thương hiệu, quy cách nên sim thường chỉ 0.4–0.75; xem C).

### C. Độ khớp nguyên liệu ↔ sản phẩm trên toàn bộ giỏ hàng

1447 cặp (nguyên liệu, sản phẩm mặc định) khác nhau, từ 13.967 dòng giỏ hàng của 1.061 món.

- sim bge-m3: trung bình **0.645**, trung vị 0.642, p10 0.513.
- Tên sản phẩm **chứa đủ các từ** của tên nguyên liệu: **49%** cặp.

| Cách ghép (`via`) | số cặp | sim trung bình | tên sản phẩm chứa đủ từ | nghi sai (sim<0.6 và không chứa từ) |
| :-- | --: | --: | --: | --: |
| sản phẩm mới đợt 2 | 564 | 0.672 | 62% | 94 |
| gộp tên bạn đã duyệt | 358 | 0.624 | 3% | 150 |
| bảng ghép 100 món (người duyệt) | 173 | 0.680 | 98% | 1 |
| AI quyết định gộp | 163 | 0.568 | 25% | 93 |
| luật khớp tên | 141 | 0.666 | 100% | 0 |
| AI: làm từ nguyên liệu khác | 30 | 0.538 | 0% | 24 |
| chọn tay (mảnh tên) | 18 | 0.595 | 22% | 9 |

**Đọc kết quả (quan trọng)**

- sim thấp **không đồng nghĩa với ghép sai.** 25 cặp "xa nhau nhất" phần lớn là ghép đúng nghĩa nhưng khác tên: bún tàu→miến, mù tạc xanh→wasabi, mì ý→mì spaghetti, tép→tôm, lạc→đậu phộng, tai vị→hoa hồi. Cột "nghi sai" (371 cặp) là cận trên, không phải số lỗi.
- Nhóm *luật khớp tên* và *bảng ghép 100 món* chứa từ gần 100% nên đáng tin; nhóm gộp tên/AI quyết định chứa từ thấp vì bản chất là đổi tên gọi.
- **Kiểm tay 55 cặp ngẫu nhiên** (14 sản phẩm mới, 14 gộp tên đã duyệt, 9 AI-gộp, 9 làm-từ-nguyên-liệu, 9 chọn tay): **khoảng 50/55 (91%) chấp nhận được**. Đây là đánh giá của AI (chưa có người kiểm), mẫu nhỏ. Các cặp có vấn đề:
  - `ớt sừng màu xanh` → ớt sừng **đỏ**.
  - `tàu hũ ky chiên` → tàu hũ chiên Ichi Sakura (khác loại).
  - `rong biển khô` → **lốc 3 gói** (sản phẩm bán thùng/lốc).
  - `tiêu tương cà` → tương cà (tên nguyên liệu bị nhiễu).
  - `nước mắm chay` → nước mắm Nam Ngư (không chay).
- Các lỗi thật còn lại: sản phẩm mặc định là kết quả khớp đầu tiên nên có thể ra dạng chế biến (thịt heo xay thay vì thịt heo); **nên cho khách đổi sản phẩm** bằng `catalog_ids` còn lại.

### Tự đánh giá mục 13

| Mục | Mức tin cậy | Lý do |
| :-- | :-: | :-- |
| Tìm đúng món | Trung bình | 22/27 top-1, 23/27 top-5, nhưng nhãn tự soạn, 27 câu, chưa so baseline |
| Ngưỡng "không tìm thấy" | Thấp | Điểm câu đúng và sai chồng lên nhau, chưa chọn được ngưỡng |
| Gõ không dấu | Thấp | Món đúng tụt hạng; chưa có đủ ca thử |
| Độ khớp giỏ hàng | Trung bình-thấp | ~91% chấp nhận được trên 55 cặp, đánh giá bởi AI, mẫu nhỏ |

## 14. Đo chính thức: Recall@5, so với ILIKE, ngưỡng "không tìm thấy" (`scripts/recipe_rag/eval_recipe_search.py`)

**Bộ câu hỏi:** `data/recipe_rag/eval/recipe_queries_v1.json`, **108 câu** gồm 87 câu có món đúng (tên món 35, biến thể cách nói 15, không dấu/gõ sai 15, theo nguyên liệu 12, câu chung 10) và 21 câu *nên từ chối* (8 món thật nhưng không có trong dữ liệu, 13 câu ngoài chủ đề).

**Cách gán nhãn:** mỗi câu có luật tìm món đúng trong 1.061 món có giỏ hàng: tên món chứa mọi cụm (`name_all`) / một cụm (`name_any`) hoặc giỏ hàng có nguyên liệu chính chứa mọi cụm (`ingr_all`), so không dấu theo từ. Script dừng nếu có câu không món nào khớp nhãn.

**Bốn cách tìm so sánh trên cùng tập:** `ILIKE nguyên câu` (SQL `name ILIKE '%câu%'`), `ILIKE từng từ` (SQL, mọi từ phải có, còn dấu), `ILIKE chuẩn hóa` (bỏ dấu + bỏ từ thừa như "món", "với", "cho", rồi mọi từ phải có trong tên, tức baseline từ khóa mạnh nhất), `bge-m3` (thẻ món + cosine), và `hybrid` (món khớp từ khóa chuẩn hóa xếp trước, sau đó món của bge-m3).

```
.venv\Scripts\python scripts\recipe_rag\eval_recipe_search.py
```

### Recall (Recall@1 / Recall@5 trên 87 câu có món đúng)

| Loại câu hỏi | n | ILIKE nguyên câu | ILIKE từng từ | ILIKE chuẩn hóa | bge-m3 | hybrid |
| :-- | --: | :-: | :-: | :-: | :-: | :-: |
| **Tất cả** | 87 | 0.48 / 0.48 | 0.51 / 0.51 | 0.89 / 0.89 | 0.91 / 0.97 | 0.95 / 0.99 |
| tên món | 35 | 1.00 / 1.00 | 1.00 / 1.00 | 1.00 / 1.00 | 1.00 / 1.00 | 1.00 / 1.00 |
| biến thể cách nói | 15 | 0.07 / 0.07 | 0.07 / 0.07 | 0.87 / 0.87 | 1.00 / 1.00 | 1.00 / 1.00 |
| không dấu / gõ sai | 15 | 0.00 / 0.00 | 0.00 / 0.00 | 0.67 / 0.67 | 0.53 / 0.80 | 0.80 / 0.93 |
| theo nguyên liệu | 12 | 0.08 / 0.08 | 0.08 / 0.08 | 0.83 / 0.83 | 0.92 / 1.00 | 0.92 / 1.00 |
| câu chung | 10 | 0.50 / 0.50 | 0.70 / 0.70 | 0.90 / 0.90 | 1.00 / 1.00 | 1.00 / 1.00 |

**Đọc kết quả**

- **bge-m3 Recall@5 = 0.97, hybrid = 0.99**; ILIKE nguyên câu / từng từ chỉ 0.48 / 0.51. Tức bge-m3 hơn rõ ILIKE thông thường (giả thuyết H5 đứng vững), nhưng **ILIKE chuẩn hóa đã đạt 0.89**, nên phần lớn lợi thế đến từ việc bỏ dấu và bỏ từ thừa chứ không riêng từ embedding.
- Khi khách gõ **đúng tên món** (35 câu) mọi cách đều 1.00: bộ này do AI viết từ tên món nên *dễ cho từ khóa*. Chỗ phân biệt là **biến thể cách nói** (bge-m3 1.00 so với ILIKE từng từ 0.07), **theo nguyên liệu** và **câu chung**.
- Câu **không dấu / gõ sai** là điểm yếu của bge-m3 (Recall@1 chỉ 0.53); nhánh từ khóa chuẩn hóa cứu được (0.80 / 0.93 với hybrid). **Nên dùng hybrid.**
- bge-m3 trượt top-5 3 câu, đều là không dấu:
  - `thit kho nuoc dua` (khong_dau_loi_go): ra Cá nục kho mía (sim 0.51)
  - `ga nuong mat ong` (khong_dau_loi_go): ra Chả cá lóc xốt mayo mật ong (sim 0.43)
  - `ca hoi kho` (khong_dau_loi_go): ra Bò kho chay (sim 0.45)
- Cảnh báo nhãn: so khớp không dấu làm *khổ* thành *kho*, *phô* thành *pho*, nên `thịt kho trứng` và `phở gà` bị luật tìm tưởng có món khớp (thực tế là *khổ qua xào trứng*, *gà chiên bắp phô mai*). Hai câu này vẫn xếp vào "nên từ chối".

### Ngưỡng "không tìm thấy"

Nhận kết quả khi sim top-1 ≥ ngưỡng, ngược lại báo không tìm thấy. "Phục vụ đúng" = câu có món đúng, được nhận và món đúng nằm trong top-5; "từ chối đúng" = câu nên từ chối bị từ chối. Ngưỡng chọn trên nửa **dev** (câu thứ lẻ mỗi loại), báo trên nửa **test** (câu chẵn).

| Ngưỡng | dev: phục vụ đúng | dev: từ chối đúng | test: phục vụ đúng | test: từ chối đúng |
| :-: | :-: | :-: | :-: | :-: |
| 0.40 | 0.96 | 0.36 | 0.98 | 0.40 |
| 0.46 | 0.93 | 0.64 | 0.95 | 0.60 |
| 0.50 | 0.93 | 0.64 | 0.93 | 0.60 |
| 0.54 | 0.89 | 0.64 | 0.88 | 0.80 |
| 0.58 | 0.82 | 0.73 | 0.81 | 0.80 |
| 0.62 | 0.78 | 0.91 | 0.76 | 1.00 |
| 0.66 | 0.64 | 1.00 | 0.64 | 1.00 |
| 0.70 | 0.44 | 1.00 | 0.52 | 1.00 |

- **bge-m3 đơn thuần:** ngưỡng chọn trên dev = **0.62**; trên test phục vụ đúng 0.76, từ chối đúng 1.00. Đổi lại phục vụ đúng bị tụt vì câu không dấu có sim thấp (trung vị 0.51) bị từ chối nhầm.
- **hybrid** (nhận nếu có món khớp từ khóa HOẶC sim ≥ ngưỡng): ngưỡng chọn trên dev = **0.66**; trên test phục vụ đúng **0.91**, từ chối đúng **0.90**.

| Ngưỡng (toàn bộ 108 câu) | bge-m3: phục vụ đúng / từ chối đúng | hybrid: phục vụ đúng / từ chối đúng |
| :-: | :-: | :-: |
| 0.50 | 0.93 / 0.62 | 0.98 / 0.62 |
| 0.56 | 0.85 / 0.76 | 0.92 / 0.71 |
| 0.60 | 0.79 / 0.86 | 0.92 / 0.81 |
| 0.62 | 0.77 / 0.95 | 0.91 / 0.86 |
| 0.66 | 0.64 / 1.00 | 0.90 / 0.91 |

**Sim top-1 theo loại câu hỏi** (cho thấy vì sao khó có một ngưỡng sạch):

| Loại | n | sim top-1 nhỏ nhất | trung vị | lớn nhất |
| :-- | --: | :-: | :-: | :-: |
| ten_mon | 35 | 0.65 | 0.73 | 0.80 |
| bien_the | 15 | 0.62 | 0.74 | 0.79 |
| khong_dau_loi_go | 15 | 0.43 | 0.51 | 0.68 |
| nguyen_lieu | 12 | 0.54 | 0.67 | 0.73 |
| chung | 10 | 0.57 | 0.64 | 0.68 |
| khong_co_mon | 8 | 0.51 | 0.59 | 0.66 |
| ngoai_chu_de | 13 | 0.33 | 0.40 | 0.46 |

- Câu ngoài chủ đề có sim ≤ 0.46 nên tách tốt. **Khó nhất là món thật nhưng không có trong dữ liệu** (sim tới 0.66, trung vị 0.59), chồng lên câu đúng dạng chung (0.57–0.68). Không ngưỡng nào tách hết được; hệ thống nên trả "món gần nhất" kèm cảnh báo thay vì khẳng định.
- Nhánh từ khóa vẫn "tìm ra" món cho 2 câu nên từ chối (thịt kho trứng, phở gà) vì chuẩn hóa dấu.

### Tự đánh giá mục 14

| Mục | Mức tin cậy | Lý do |
| :-- | :-: | :-- |
| Recall@5 bge-m3 = 0.97 | Trung bình-thấp | 87 câu, nhãn **do AI soạn từ tên món** nên lạc quan, đặc biệt nhóm "tên món"; chưa có truy vấn thật của khách |
| So với ILIKE (H5) | Trung bình | Hơn ILIKE thông thường rất rõ; hơn ILIKE chuẩn hóa chỉ ~0.08; chênh lệch trên vài câu nên chưa đủ để kết luận chắc |
| Ngưỡng | Thấp | 21 câu từ chối, chia đôi dev/test còn ~10 câu mỗi bên, nên giá trị ngưỡng dao động theo tập; xem như khoảng 0.60–0.66, không phải con số chốt |

**Việc nên làm tiếp:** thu 50–100 câu hỏi thật (từ người trong nhóm hoặc log), nhờ người gán nhãn độc lập; thử `unaccent` trong Postgres cho nhánh từ khóa; xem xét cho `hybrid` dùng điểm gộp (RRF) thay vì xếp lần lượt.

### Tổ chức lại thư mục theo module

Scripts, dữ liệu, notebook đã chuyển vào `scripts/recipe_rag/`, `data/recipe_rag/`, `notebooks/recipe_rag/` để nhánh `predictive-reorder` (LightGBM) dùng `.../reorder/` song song, không đè nhau. Quy ước và bảng chuyển file cho nhánh kia: `docs/STRUCTURE.md`. Lệnh trong các mục trước đã được sửa đường dẫn theo cấu trúc mới.

## 15. ai-service: lõi tìm món, giỏ hàng và lớp lời AI (GĐ 1–4)

Kế hoạch đầy đủ, hợp đồng JSON và các giai đoạn: `docs/recipe_rag/plan-ai-copilot.md`. Mục này ghi phần đã làm: **GĐ 1–4 (lõi, chưa có route HTTP)**. GĐ 5–12 hoãn đến khi web bán hàng cơ bản hoàn chỉnh (mục 16).

**Nguyên tắc:** `data` (món + giỏ) luôn do **code** dựng; mô hình ngôn ngữ chỉ viết phần lời (`ai`), và lỗi ở lớp lời không bao giờ làm hỏng giỏ hàng.

```
câu hỏi ─► RecipeService.search ─► data (5 món + giỏ hai phần)      ◄── luôn có
                          └─► CopilotService ─► provider(Gemini/Claude) ─► check_answer ─► ai   ◄── có thể None
```

### 15.1 Dữ liệu cho service (GĐ 1)
- Hai bảng mới trong `ai_db`: `catalog_product` (2.129 sản phẩm) và `recipe_basket` (1.061 món, đều đủ nguyên liệu chính). Migration `db/migrations/002_*.sql` (cho DB đã có) và `db/init/03_*.sql` (cho DB mới); nạp bằng `load_basket_data.py` (chạy lại an toàn, có `--dry-run`).
- Giá, ảnh, tồn kho **không** nằm ở `ai_db`: do Medusa quản lý, ghép theo `catalog_id`.
- Khung `src/`: `config/settings.py` (đọc `.env`, giấu mật khẩu và khóa API), `ml/embedder.py` (nạp bge-m3 một lần, fp16 trên GPU), `utils/text_cleaner.py`, `schemas/*_dto.py`.

### 15.2 Tìm món và giỏ hàng (GĐ 2, `src/services/recipe_service.py`)
- **Tìm món (hybrid):** món có tên chứa mọi từ khóa (đã bỏ dấu, bỏ từ thừa) xếp trước, rồi tới món của bge-m3 trên thẻ món. Kết quả được **nhận** nếu có món khớp từ khóa hoặc sim top-1 ≥ `search_threshold` (mặc định 0.64). Không nhận thì vẫn trả món gần nhất, kèm `accepted=false`.
- **Giỏ hàng (`build_basket`, hàm thuần, có test):**
  - Nguyên liệu chính (`core`) → `buy`, số lượng mặc định 1; đồ trong tủ bếp (`addon`) và đồ khách nói đã có (`have`, theo tên hoặc `catalog_id`) → `maybe_have`, mặc định 0.
  - Lượng cần = lượng trong công thức × (số người / khẩu phần công thức). `pack_hint` = làm tròn lên (lượng cần / quy cách) **chỉ khi** cùng loại đơn vị (g↔kg hoặc ml↔lít); ngược lại để trống thay vì đoán.
  - Cùng nguyên liệu nhiều dòng thì cộng dồn nếu cùng đơn vị. Dòng ăn kèm, dòng bỏ khỏi giỏ và sản phẩm không còn trong catalog không vào giỏ.
- **Kiểm bằng bộ 108 câu qua chính service** (`check_search_service.py`):

| Chỉ số | Kết quả |
| :-- | :-: |
| Recall@1 / Recall@5 | 0.95 / **0.99** (khớp số đo hybrid ở mục 14) |
| Phục vụ đúng (có món đúng và được nhận) | 0.90 |
| Từ chối đúng (21 câu nên từ chối) | 0.86 (18/21) |
| Độ trễ p50 / p95 / max | 46 / 83 / 103 ms (tiêu chí: p95 < 1 giây) |
| Khởi động (nạp model + 1.061 món + 2.129 sản phẩm) | khoảng 8 giây |
| Giỏ sai quy tắc số lượng mặc định | không |

- 3 câu bị nhận nhầm: `bò xào hành tây` (sim 0.66, gần ngưỡng), `thịt kho trứng` và `phở gà` (do bỏ dấu làm *khổ*→*kho*, *phô*→*pho*).

### 15.3 Dữ kiện và prompt (GĐ 3, `copilot_facts.py`)
- Prompt gồm câu hỏi (đặt trong `<<< >>>` như dữ liệu), số người, và **3 món ứng viên** (`copilot_candidates`), mỗi món có mô tả, các bước rút gọn và giỏ hai phần.
- Mỗi dòng nguyên liệu mang số liệu đã tính sẵn: `cần ~500 g cho 4 người | sản phẩm: … (300 g) | gợi ý ~2 gói`; nguyên liệu tự làm được đánh dấu. Không có số thì ghi "không quy được ra số gói" để mô hình không bịa.
- **Mô hình không thấy `catalog_id`**, nên không thể ghi id vào giỏ. Xem prompt thật: `try_copilot.py "..." --prompt` (không gọi mô hình).

### 15.4 Lớp lời AI (GĐ 4, `copilot_service.py`)
- **Schema mô hình phải trả** (`AiAnswer`, cố ý nhỏ, không có giá trị mặc định): `recipe_id`, `explanation`, `item_notes[]`, `prep_notes[]`, `follow_up_question`.
- **Kiểm tra bằng dữ liệu thật (`check_answer`):** `recipe_id` phải là một trong các món ứng viên; `explanation` không rỗng, ≤ 800 ký tự (sai → `ProviderInvalid`, thử lại **một lần** kèm lý do); ghi chú nhắc nguyên liệu không có trong giỏ bị **bỏ riêng ghi chú đó**; `prep_notes` chỉ nhận cho nguyên liệu `derived`.
- **Dự phòng theo lớp:** thử nhà cung cấp bật theo thứ tự ưu tiên; hết giờ thì **không thử lại** cùng nhà cung cấp mà sang nhà cung cấp kế; tổng thời gian có trần (`ai_total_budget_s`). Mọi lỗi chỉ làm `ai = null`.

| `ai_status` | Ý nghĩa |
| :-- | :-- |
| `ok` | có lớp lời hợp lệ |
| `skipped_no_match` | `data.accepted = false`: không nhờ mô hình nói về món không đúng ý |
| `fallback_disabled` | mọi nhà cung cấp đang tắt |
| `fallback_invalid` / `fallback_timeout` / `fallback_error` | trạng thái của nhà cung cấp thử sau cùng |

- **Nhà cung cấp** (`gemini_service.py`, `claude_service.py`, chung giao diện `llm_provider.py`): Gemini dùng `response_schema` với model Pydantic; Claude Haiku 5.5 (`claude-haiku-5-5`) dùng `messages.parse` với `effort` thấp và `max_retries=0` để việc thử lại do `CopilotService` quyết. Cả hai đổi lỗi mạng/API thành `ProviderError` gọn (đã thử bằng khóa giả: Gemini 400, Claude 401).
- **Bật/tắt và ưu tiên** hiện đọc từ `.env` (`AI_GEMINI_ENABLED`, `AI_CLAUDE_ENABLED`, `*_PRIORITY`). Gemini là chính, Claude tắt mặc định (bật khi demo). GĐ 5 sẽ chuyển sang bảng trong `ai_db` cho trang admin.
- **Mỗi lần gọi** tạo một `CallRecord` (nhà cung cấp, trạng thái, độ trễ, token); hiện ghi vào log, GĐ 6 sẽ ghi vào bảng.

### 15.5 Kiểm thử
- `pytest tests/recipe_rag`: **29 test đều qua**: chuẩn hóa văn bản, `pack_hint` và quy tắc giỏ, tích hợp `RecipeService` với `ai_db` thật (5 test), và 14 test lớp lời bằng nhà cung cấp giả: thành công, JSON sai thì thử lại kèm lý do, hết giờ thì sang nhà cung cấp khác, tắt hết, đổi thứ tự ưu tiên, bỏ ghi chú sai, không nhờ mô hình khi không có món đúng.
- Thử tay với khóa thật: `try_copilot.py "gà nướng mật ong cho 4 người" --providers gemini` (hoặc `claude`, hoặc `gemini,claude`). Cần `GEMINI_API_KEY` / `ANTHROPIC_API_KEY` trong `.env`; mỗi lần chạy tốn một ít hạn mức.

In [ ]:
# Kiểm service bằng bộ 108 câu (cần ai_db đang chạy)
# .venv\Scripts\python scripts\recipe_rag\check_search_service.py

# Chạy toàn bộ test (cài một lần: pip install -r requirements-dev.txt)
# .venv\Scripts\python -m pytest tests/recipe_rag -q

# Xem prompt gửi cho mô hình, KHÔNG gọi mô hình
# .venv\Scripts\python scripts\recipe_rag\try_copilot.py "gà nướng mật ong cho 4 người" --prompt

# Gọi mô hình thật (cần khóa trong .env)
# .venv\Scripts\python scripts\recipe_rag\try_copilot.py "canh chua cá lóc" --providers gemini

### Tự đánh giá mục 15

| Mục | Mức tin cậy | Lý do |
| :-- | :-: | :-- |
| Tìm món và dựng giỏ (lõi) | Trung bình | Recall@5 0.99 và p95 83 ms đo qua chính service; nhưng nhãn do AI soạn và ngưỡng chưa chốt (mục 14) |
| Quy tắc giỏ (hai phần, số lượng, `pack_hint`) | Trung bình-cao | Hàm thuần, có test; nhưng sản phẩm mặc định là kết quả khớp đầu tiên nên có thể ra dạng chế biến (mục 13) |
| Lớp kiểm tra và dự phòng | Trung bình | Logic được test bằng nhà cung cấp giả; **chưa gọi mô hình thật** nên chưa biết tỉ lệ JSON hợp lệ, độ trễ, chất lượng lời giải thích |
| Provider Gemini / Claude | Thấp-trung bình | Đã đối chiếu tài liệu SDK và thử lỗi bằng khóa giả; lỗi 401 của Claude xảy ra trước bước kiểm tra schema nên chưa chứng minh schema được chấp nhận; tên model Gemini (`gemini-flash-latest`) chưa chốt |
| Prompt | Thấp | Chưa so với phiên bản khác, chưa đo trên 108 câu |

**Chưa làm (cố ý hoãn):** route HTTP, cấu hình bật/tắt trong `ai_db`, bảng log, đo hai nhà cung cấp trên 108 câu, tích hợp backend/frontend/admin.

## 16. Các bước tiếp theo

**Đã xong từ kế hoạch ban đầu:** chuẩn hóa dữ liệu, ghép nguyên liệu → sản phẩm, khóa `catalog_id`, mở rộng catalog cho món phổ biến, bộ nhãn 108 câu, đo Recall@5 và so với ILIKE, chọn ngưỡng, `ai_db`, lõi tìm món + giỏ + lớp lời AI.

**Hoãn đến khi web bán hàng cơ bản hoàn chỉnh** (các bước này cần Medusa có catalog, giá, giỏ hàng; xem `docs/recipe_rag/plan-ai-copilot.md`, GĐ 5–12):
- GĐ 5–6: cấu hình bật/tắt nhà cung cấp trong `ai_db`, bảng log và thống kê.
- GĐ 7: đo Gemini và Claude trên bộ 108 câu + 50 câu hỏi thật (tỉ lệ JSON hợp lệ, tỉ lệ dự phòng, độ trễ, chi phí).
- GĐ 8: route FastAPI (`/health`, `/v1/search`, `/v1/copilot`, `/admin/*`), README `ai-service`, `.env.example`, Dockerfile.
- GĐ 9–12: route backend + workflow thêm vào giỏ trong Medusa, giao diện RAG trên frontend, trang admin bật/tắt model, thử trọn luồng và chuẩn bị demo (bật Claude).

**Việc có thể làm trong lúc chờ:**
- Thêm khóa vào `.env` và chạy `try_copilot.py` với Gemini rồi Claude để xem lời giải thích thật.
- Xác nhận 373 sản phẩm thêm tay (tên, đơn vị, danh mục): đổi tên/đơn vị **không làm đổi `catalog_id`**; chạy lại `build_final_handoff_v2.py`.
- Thu 50–100 câu hỏi thật, nhờ người gán nhãn độc lập; thử `unaccent` trong Postgres; thử gộp điểm (RRF) thay vì xếp lần lượt.
- Chọn sản phẩm mặc định tốt hơn (ưu tiên hàng tươi, tránh lốc/thùng và dạng chế biến) khi matching đủ tốt để làm nhóm sản phẩm thay thế.

**Hoãn lâu hơn:** dị ứng (hướng rẻ: cảnh báo theo từ khóa nguyên liệu), ngân sách (mô hình hiểu câu, code cộng giá từ web), loại trừ đồ người dùng đã có ở mức hội thoại nhiều lượt.

## 17. Tổng kết đánh giá

| Mục | Mức tin cậy | Lý do |
| :-- | :-: | :-- |
| Chuẩn hóa tên công thức | Trung bình | Số liệu trước/sau rõ, nhưng chỉ kiểm bằng mắt trên mẫu, chưa có tập nhãn |
| Chuẩn hóa catalog | Trung bình-thấp | Mới xem ~22 mẫu, `brand` là đoán |
| Độ phủ 69% *(lịch sử, ở giai đoạn đầu)* | Thấp (cận trên) | Gộp khớp đúng và khớp tiền tố, chưa kiểm đúng/sai, chưa phải độ phủ theo công thức; chỉ 125/654 tên khớp đúng |
| bge-m3: gợi ý gộp tên | Dùng được để *gợi ý* | Cặp đầu đúng nhưng similarity cao vẫn có cặp sai, bắt buộc duyệt tay |
| bge-m3: ghép → catalog | Thấp, không dùng một mình | 447/654 nguyên liệu không có khớp chữ và top-1 thường sai; similarity không phân biệt được "có" và "không có" |
| Tập 100 món (≈49 đủ + 51 thiếu 1) *(lịch sử)* | Trung bình | Luật khớp đã chặn các lỗi thấy được, nhưng vẫn là luật, chưa duyệt tay |
| Quy tắc số lượng (1 gói, `packs > 1` khi vượt quy cách) | Trung bình | Đã cài trong `build_final_handoff.py`; đo trên dữ liệu thì 10/150 dòng cần hơn 1 gói; bỏ bảng quy đổi đơn vị |
| `ai_db` + GPU + tìm kiếm | Trung bình | Dữ liệu thật đã nạp và tìm được đúng món trên các câu hỏi thử; số đo GPU tái lập được; Recall@5 và ngưỡng đã đo ở mục 14 |
| Giỏ hàng gắn với kết quả tìm kiếm | Trung bình | *(Cập nhật)* 1.061 món có giỏ hàng (mục 12), tìm qua service cho Recall@5 0.99; xem mục 15.2. Bản cũ chỉ có 100 món không phổ biến |
| Mở rộng catalog đợt 2 (mục 12) | Trung bình | Kiểm tự động 0 lỗi, id cũ giữ nguyên; thêm/gộp do AI quyết, người chỉ duyệt tầng cần quyết |
| Thử RAG đầu-cuối (mục 13) | Trung bình-thấp | 22/27 top-1; giỏ hàng ~91% chấp nhận trên mẫu 55 cặp do AI chấm; chưa có Recall@5 chính thức |
| Đo chính thức tìm món (mục 14) | Trung bình-thấp | Recall@5 bge-m3 0.97, hybrid 0.99, ILIKE chuẩn hóa 0.89; nhãn do AI soạn; ngưỡng chưa chốt |
| ai-service: lõi tìm món + giỏ + lớp lời AI (mục 15) | Trung bình | Recall@5 0.99, p95 83 ms, 29 test qua; **chưa gọi mô hình thật**, prompt chưa đo |

**Điểm yếu lớn nhất của pipeline hiện tại:** *nhãn đánh giá do AI soạn, chưa có truy vấn thật và người gán nhãn độc lập*. Có bộ 108 câu (mục 14) nhưng kết quả có xu hướng lạc quan; ghép nguyên liệu → sản phẩm mới có 55 cặp được kiểm (khoảng 91% chấp nhận được). Nên có tập nhãn độc lập trước khi chốt ngưỡng hay đánh giá lớp lời AI.

**Phụ thuộc vào dữ liệu:** nhiều nguyên liệu chưa khớp là do catalog thiếu hàng (đặc biệt đồ tươi). Nếu số công thức đủ nguyên liệu thấp, nguyên nhân có thể nằm ở catalog chứ không phải pipeline.

**Lưu ý đạo văn và demo:** thầy sẽ hỏi từng dòng code của tính năng demo. Giữ luật đơn giản, đặt tên rõ, và phần văn xuôi nộp cho thầy cần nhóm tự viết lại.
